# Chapter 19: Parser

> *"The most important asset of a software engineer is the ability to
> structure complex systems hierarchically. Top-down parsing reflects
> this principle with crystalline clarity: from a single root
> proposition, an entire universe of meaning unfolds."*  
> — **Niklaus Wirth**, *Algorithms + Data Structures = Programs* (1976)

------------------------------------------------------------------------

### 19.1 From Linear Sequences to Hierarchical Trees

In **Chapter 18**, we constructed the compiler's lexical front-end. Our
scanner consumed raw source text and produced a flat, linear stream of
typed tokens:

    [ Keyword_Let, Identifier("x"), Equal, Literal_Int(10), Plus, Literal_Int(20), Semicolon ]

While this linear stream classifies individual words and symbols, it
possesses no structural understanding. It does not know that `10 + 20`
forms a single arithmetic expression, that `+` binds tighter than
assignment, or that nested curly braces `{ ... }` delineate lexical
scopes.

The task of **Syntactic Analysis (Parsing)** is to transform this flat
token sequence into a rich, recursive graph called the **Abstract Syntax
Tree (AST)**.

<figure id="fig:concrete-abstract">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI1NThwdCIgaGVpZ2h0PSIyMThwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCA1NTguMDAgMjE4LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIx
NCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gYXNzaWdubWVudCAtLT4KPGcgaWQ9Im5vZGUxIiBj
bGFzcz0ibm9kZSI+Cjx0aXRsZT5hc3NpZ25tZW50PC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9u
ZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxNDksLTIxMCA2MCwtMjEwIDYwLC0xNzQgMTQ5LC0x
NzQgMTQ5LC0yMTAiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTA0LjUiIHk9Ii0x
ODkuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3Bh
Y2UiIGZvbnQtc2l6ZT0iMTAuMDAiPiZsdDtBc3NpZ25tZW50Jmd0OzwvdGV4dD4KPC9nPgo8IS0t
IGlkIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmlkPC90aXRsZT4KPHBv
bHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSI1OSwtMTUyIDAsLTE1MiAw
LC0xMTYgNTksLTExNiA1OSwtMTUyIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjI5
LjUiIHk9Ii0xMzEuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25v
LCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPklEKCZxdW90O3gmcXVvdDspPC90ZXh0Pgo8
L2c+CjwhLS0gYXNzaWdubWVudCYjNDU7Jmd0O2lkIC0tPgo8ZyBpZD0iZWRnZTEiIGNsYXNzPSJl
ZGdlIj4KPHRpdGxlPmFzc2lnbm1lbnQmIzQ1OyZndDtpZDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5v
bmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik04MS41OSwtMTczLjg5Qzc0LjksLTE2OC45IDY3LjQ4LC0x
NjMuMzYgNjAuMzksLTE1OC4wNiIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFj
ayIgcG9pbnRzPSI2Mi4zOCwtMTU1LjE4IDUyLjI3LC0xNTIgNTguMTksLTE2MC43OSA2Mi4zOCwt
MTU1LjE4Ii8+CjwvZz4KPCEtLSBlcXVhbCAtLT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9kZSI+
Cjx0aXRsZT5lcXVhbDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2si
IHBvaW50cz0iMTMxLjUsLTE1MiA3Ny41LC0xNTIgNzcuNSwtMTE2IDEzMS41LC0xMTYgMTMxLjUs
LTE1MiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxMDQuNSIgeT0iLTEzMS41IiBm
b250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9u
dC1zaXplPSIxMC4wMCI+JiMzOTs9JiMzOTs8L3RleHQ+CjwvZz4KPCEtLSBhc3NpZ25tZW50JiM0
NTsmZ3Q7ZXF1YWwgLS0+CjxnIGlkPSJlZGdlMiIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+YXNzaWdu
bWVudCYjNDU7Jmd0O2VxdWFsPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFj
ayIgZD0iTTEwNC41LC0xNzMuODlDMTA0LjUsLTE3MC4yIDEwNC41LC0xNjYuMjEgMTA0LjUsLTE2
Mi4yNCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxMDgs
LTE2MiAxMDQuNSwtMTUyIDEwMSwtMTYyIDEwOCwtMTYyIi8+CjwvZz4KPCEtLSBleHByIC0tPgo8
ZyBpZD0ibm9kZTQiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmV4cHI8L3RpdGxlPgo8cG9seWdvbiBm
aWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjIwMy41LC0xNTIgMTQ5LjUsLTE1MiAx
NDkuNSwtMTE2IDIwMy41LC0xMTYgMjAzLjUsLTE1MiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIiB4PSIxNzYuNSIgeT0iLTEzMS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFW
dSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+Jmx0O0V4cHImZ3Q7PC90
ZXh0Pgo8L2c+CjwhLS0gYXNzaWdubWVudCYjNDU7Jmd0O2V4cHIgLS0+CjxnIGlkPSJlZGdlMyIg
Y2xhc3M9ImVkZ2UiPgo8dGl0bGU+YXNzaWdubWVudCYjNDU7Jmd0O2V4cHI8L3RpdGxlPgo8cGF0
aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTI2LjQ5LC0xNzMuODlDMTMyLjc3LC0x
NjkuMDEgMTM5LjczLC0xNjMuNiAxNDYuNDEsLTE1OC40MSIvPgo8cG9seWdvbiBmaWxsPSJibGFj
ayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxNDguOSwtMTYwLjkgMTU0LjY0LC0xNTIgMTQ0LjYs
LTE1NS4zOCAxNDguOSwtMTYwLjkiLz4KPC9nPgo8IS0tIGludDEgLS0+CjxnIGlkPSJub2RlNSIg
Y2xhc3M9Im5vZGUiPgo8dGl0bGU+aW50MTwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0
cm9rZT0iYmxhY2siIHBvaW50cz0iMTMxLjUsLTk0IDc3LjUsLTk0IDc3LjUsLTU4IDEzMS41LC01
OCAxMzEuNSwtOTQiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTA0LjUiIHk9Ii03
My41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFj
ZSIgZm9udC1zaXplPSIxMC4wMCI+SU5UKDEpPC90ZXh0Pgo8L2c+CjwhLS0gZXhwciYjNDU7Jmd0
O2ludDEgLS0+CjxnIGlkPSJlZGdlNCIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+ZXhwciYjNDU7Jmd0
O2ludDE8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTU0LjUx
LC0xMTUuODlDMTQ4LjIzLC0xMTEuMDEgMTQxLjI3LC0xMDUuNiAxMzQuNTksLTEwMC40MSIvPgo8
cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxMzYuNCwtOTcuMzgg
MTI2LjM2LC05NCAxMzIuMSwtMTAyLjkgMTM2LjQsLTk3LjM4Ii8+CjwvZz4KPCEtLSBwbHVzIC0t
Pgo8ZyBpZD0ibm9kZTYiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnBsdXM8L3RpdGxlPgo8cG9seWdv
biBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjIwMy41LC05NCAxNDkuNSwtOTQg
MTQ5LjUsLTU4IDIwMy41LC01OCAyMDMuNSwtOTQiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRs
ZSIgeD0iMTc2LjUiIHk9Ii03My41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBT
YW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+JiMzOTsrJiMzOTs8L3RleHQ+
CjwvZz4KPCEtLSBleHByJiM0NTsmZ3Q7cGx1cyAtLT4KPGcgaWQ9ImVkZ2U1IiBjbGFzcz0iZWRn
ZSI+Cjx0aXRsZT5leHByJiM0NTsmZ3Q7cGx1czwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0
cm9rZT0iYmxhY2siIGQ9Ik0xNzYuNSwtMTE1Ljg5QzE3Ni41LC0xMTIuMiAxNzYuNSwtMTA4LjIx
IDE3Ni41LC0xMDQuMjQiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBv
aW50cz0iMTgwLC0xMDQgMTc2LjUsLTk0IDE3MywtMTA0IDE4MCwtMTA0Ii8+CjwvZz4KPCEtLSB0
ZXJtIC0tPgo8ZyBpZD0ibm9kZTciIGNsYXNzPSJub2RlIj4KPHRpdGxlPnRlcm08L3RpdGxlPgo8
cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjI3NS41LC05NCAyMjEu
NSwtOTQgMjIxLjUsLTU4IDI3NS41LC01OCAyNzUuNSwtOTQiLz4KPHRleHQgdGV4dC1hbmNob3I9
Im1pZGRsZSIgeD0iMjQ4LjUiIHk9Ii03My41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERl
amFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+Jmx0O1Rlcm0mZ3Q7
PC90ZXh0Pgo8L2c+CjwhLS0gZXhwciYjNDU7Jmd0O3Rlcm0gLS0+CjxnIGlkPSJlZGdlNiIgY2xh
c3M9ImVkZ2UiPgo8dGl0bGU+ZXhwciYjNDU7Jmd0O3Rlcm08L3RpdGxlPgo8cGF0aCBmaWxsPSJu
b25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTk4LjQ5LC0xMTUuODlDMjA0Ljc3LC0xMTEuMDEgMjEx
LjczLC0xMDUuNiAyMTguNDEsLTEwMC40MSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tl
PSJibGFjayIgcG9pbnRzPSIyMjAuOSwtMTAyLjkgMjI2LjY0LC05NCAyMTYuNiwtOTcuMzggMjIw
LjksLTEwMi45Ii8+CjwvZz4KPCEtLSBpbnQyIC0tPgo8ZyBpZD0ibm9kZTgiIGNsYXNzPSJub2Rl
Ij4KPHRpdGxlPmludDI8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNr
IiBwb2ludHM9IjIwMy41LC0zNiAxNDkuNSwtMzYgMTQ5LjUsMCAyMDMuNSwwIDIwMy41LC0zNiIv
Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxNzYuNSIgeT0iLTE1LjUiIGZvbnQtZmFt
aWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9
IjEwLjAwIj5JTlQoMik8L3RleHQ+CjwvZz4KPCEtLSB0ZXJtJiM0NTsmZ3Q7aW50MiAtLT4KPGcg
aWQ9ImVkZ2U3IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT50ZXJtJiM0NTsmZ3Q7aW50MjwvdGl0bGU+
CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0yMjYuNTEsLTU3Ljg5QzIyMC4y
MywtNTMuMDEgMjEzLjI3LC00Ny42IDIwNi41OSwtNDIuNDEiLz4KPHBvbHlnb24gZmlsbD0iYmxh
Y2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjA4LjQsLTM5LjM4IDE5OC4zNiwtMzYgMjA0LjEs
LTQ0LjkgMjA4LjQsLTM5LjM4Ii8+CjwvZz4KPCEtLSBzdGFyIC0tPgo8ZyBpZD0ibm9kZTkiIGNs
YXNzPSJub2RlIj4KPHRpdGxlPnN0YXI8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJv
a2U9ImJsYWNrIiBwb2ludHM9IjI3NS41LC0zNiAyMjEuNSwtMzYgMjIxLjUsMCAyNzUuNSwwIDI3
NS41LC0zNiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIyNDguNSIgeT0iLTE1LjUi
IGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBm
b250LXNpemU9IjEwLjAwIj4mIzM5O++8iiYjMzk7PC90ZXh0Pgo8L2c+CjwhLS0gdGVybSYjNDU7
Jmd0O3N0YXIgLS0+CjxnIGlkPSJlZGdlOCIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+dGVybSYjNDU7
Jmd0O3N0YXI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMjQ4
LjUsLTU3Ljg5QzI0OC41LC01NC4yIDI0OC41LC01MC4yMSAyNDguNSwtNDYuMjQiLz4KPHBvbHln
b24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjUyLC00NiAyNDguNSwtMzYg
MjQ1LC00NiAyNTIsLTQ2Ii8+CjwvZz4KPCEtLSBpbnQzIC0tPgo8ZyBpZD0ibm9kZTEwIiBjbGFz
cz0ibm9kZSI+Cjx0aXRsZT5pbnQzPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tl
PSJibGFjayIgcG9pbnRzPSIzNDcuNSwtMzYgMjkzLjUsLTM2IDI5My41LDAgMzQ3LjUsMCAzNDcu
NSwtMzYiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMzIwLjUiIHk9Ii0xNS41IiBm
b250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9u
dC1zaXplPSIxMC4wMCI+SU5UKDMpPC90ZXh0Pgo8L2c+CjwhLS0gdGVybSYjNDU7Jmd0O2ludDMg
LS0+CjxnIGlkPSJlZGdlOSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+dGVybSYjNDU7Jmd0O2ludDM8
L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMjcwLjQ5LC01Ny44
OUMyNzYuNzcsLTUzLjAxIDI4My43MywtNDcuNiAyOTAuNDEsLTQyLjQxIi8+Cjxwb2x5Z29uIGZp
bGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjI5Mi45LC00NC45IDI5OC42NCwtMzYg
Mjg4LjYsLTM5LjM4IDI5Mi45LC00NC45Ii8+CjwvZz4KPCEtLSBsZXRzdG10IC0tPgo8ZyBpZD0i
bm9kZTExIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5sZXRzdG10PC90aXRsZT4KPHBvbHlnb24gZmls
bD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSI0NzAsLTIxMCAzNzUsLTIxMCAzNzUsLTE3
NCA0NzAsLTE3NCA0NzAsLTIxMCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI0MjIu
NSIgeT0iLTE4OS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8s
IG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+TGV0U3RtdCAoJnF1b3Q7eCZxdW90Oyk8L3Rl
eHQ+CjwvZz4KPCEtLSBiaW5leHByMSAtLT4KPGcgaWQ9Im5vZGUxMiIgY2xhc3M9Im5vZGUiPgo8
dGl0bGU+YmluZXhwcjE8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNr
IiBwb2ludHM9IjQ3MywtMTUyIDM3MiwtMTUyIDM3MiwtMTE2IDQ3MywtMTE2IDQ3MywtMTUyIi8+
Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjQyMi41IiB5PSItMTMxLjUiIGZvbnQtZmFt
aWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9
IjEwLjAwIj5CaW5hcnlFeHByICgrKTwvdGV4dD4KPC9nPgo8IS0tIGxldHN0bXQmIzQ1OyZndDti
aW5leHByMSAtLT4KPGcgaWQ9ImVkZ2UxMCIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+bGV0c3RtdCYj
NDU7Jmd0O2JpbmV4cHIxPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
ZD0iTTQyMi41LC0xNzMuODlDNDIyLjUsLTE3MC4yIDQyMi41LC0xNjYuMjEgNDIyLjUsLTE2Mi4y
NCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSI0MjYsLTE2
MiA0MjIuNSwtMTUyIDQxOSwtMTYyIDQyNiwtMTYyIi8+CjwvZz4KPCEtLSBsaXRlcmFsMSAtLT4K
PGcgaWQ9Im5vZGUxMyIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+bGl0ZXJhbDE8L3RpdGxlPgo8cG9s
eWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjQwOSwtOTQgMzI2LC05NCAz
MjYsLTU4IDQwOSwtNTggNDA5LC05NCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIz
NjcuNSIgeT0iLTczLjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9u
bywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5MaXRlcmFsICgxKTwvdGV4dD4KPC9nPgo8
IS0tIGJpbmV4cHIxJiM0NTsmZ3Q7bGl0ZXJhbDEgLS0+CjxnIGlkPSJlZGdlMTEiIGNsYXNzPSJl
ZGdlIj4KPHRpdGxlPmJpbmV4cHIxJiM0NTsmZ3Q7bGl0ZXJhbDE8L3RpdGxlPgo8cGF0aCBmaWxs
PSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNNDA1LjcsLTExNS44OUM0MDEuMjIsLTExMS4zMyAz
OTYuMjksLTEwNi4zMiAzOTEuNTEsLTEwMS40NSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ry
b2tlPSJibGFjayIgcG9pbnRzPSIzOTMuNywtOTguNjggMzg0LjIsLTk0IDM4OC43MSwtMTAzLjU5
IDM5My43LC05OC42OCIvPgo8L2c+CjwhLS0gYmluZXhwcjIgLS0+CjxnIGlkPSJub2RlMTQiIGNs
YXNzPSJub2RlIj4KPHRpdGxlPmJpbmV4cHIyPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIg
c3Ryb2tlPSJibGFjayIgcG9pbnRzPSI1MjgsLTk0IDQyNywtOTQgNDI3LC01OCA1MjgsLTU4IDUy
OCwtOTQiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNDc3LjUiIHk9Ii03My41IiBm
b250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9u
dC1zaXplPSIxMC4wMCI+QmluYXJ5RXhwciAoKik8L3RleHQ+CjwvZz4KPCEtLSBiaW5leHByMSYj
NDU7Jmd0O2JpbmV4cHIyIC0tPgo8ZyBpZD0iZWRnZTEyIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5i
aW5leHByMSYjNDU7Jmd0O2JpbmV4cHIyPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tl
PSJibGFjayIgZD0iTTQzOS4zLC0xMTUuODlDNDQzLjc4LC0xMTEuMzMgNDQ4LjcxLC0xMDYuMzIg
NDUzLjQ5LC0xMDEuNDUiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBv
aW50cz0iNDU2LjI5LC0xMDMuNTkgNDYwLjgsLTk0IDQ1MS4zLC05OC42OCA0NTYuMjksLTEwMy41
OSIvPgo8L2c+CjwhLS0gbGl0ZXJhbDIgLS0+CjxnIGlkPSJub2RlMTUiIGNsYXNzPSJub2RlIj4K
PHRpdGxlPmxpdGVyYWwyPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFj
ayIgcG9pbnRzPSI0NDksLTM2IDM2NiwtMzYgMzY2LDAgNDQ5LDAgNDQ5LC0zNiIvPgo8dGV4dCB0
ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI0MDcuNSIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJDb3Vy
aWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5M
aXRlcmFsICgyKTwvdGV4dD4KPC9nPgo8IS0tIGJpbmV4cHIyJiM0NTsmZ3Q7bGl0ZXJhbDIgLS0+
CjxnIGlkPSJlZGdlMTMiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmJpbmV4cHIyJiM0NTsmZ3Q7bGl0
ZXJhbDI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNNDU2LjEy
LC01Ny44OUM0NTAuMDEsLTUzLjAxIDQ0My4yNSwtNDcuNiA0MzYuNzYsLTQyLjQxIi8+Cjxwb2x5
Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjQzOC43NSwtMzkuNTIgNDI4
Ljc1LC0zNiA0MzQuMzcsLTQ0Ljk4IDQzOC43NSwtMzkuNTIiLz4KPC9nPgo8IS0tIGxpdGVyYWwz
IC0tPgo8ZyBpZD0ibm9kZTE2IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5saXRlcmFsMzwvdGl0bGU+
Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iNTUwLC0zNiA0Njcs
LTM2IDQ2NywwIDU1MCwwIDU1MCwtMzYiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0i
NTA4LjUiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1v
bm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+TGl0ZXJhbCAoMyk8L3RleHQ+CjwvZz4K
PCEtLSBiaW5leHByMiYjNDU7Jmd0O2xpdGVyYWwzIC0tPgo8ZyBpZD0iZWRnZTE0IiBjbGFzcz0i
ZWRnZSI+Cjx0aXRsZT5iaW5leHByMiYjNDU7Jmd0O2xpdGVyYWwzPC90aXRsZT4KPHBhdGggZmls
bD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTQ4Ni45NywtNTcuODlDNDg5LjI1LC01My43NyA0
OTEuNzQsLTQ5LjI3IDQ5NC4xOSwtNDQuODQiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9r
ZT0iYmxhY2siIHBvaW50cz0iNDk3LjMxLC00Ni40NSA0OTkuMDksLTM2IDQ5MS4xOCwtNDMuMDYg
NDk3LjMxLC00Ni40NSIvPgo8L2c+CjwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 19–1: Comparison of Parse Tree and AST for expression
"let x = 1 + 2 * 3;"</figcaption>
</figure>

#### 19.1.1 Concrete Syntax Trees versus Abstract Syntax Trees

- **Concrete Syntax Tree (CST / Parse Tree):** A direct 1:1 graphical
  representation of the formal grammar productions. A CST retains every
  semicolon, comma, and grouping parenthesis. While useful in
  theoretical linguistics, CSTs are bloated and tedious for compilation
  passes.
- **Abstract Syntax Tree (AST):** An optimised, structural
  representation of program semantics. Syntactic trivia (such as
  semicolons and grouping parentheses) are discarded. Operator nodes
  directly link their operand subtrees. The AST is the central data
  structure around which type checking, optimisation, and code emission
  operate.

See Figure 19–1 for a visual comparison of the two.

### 19.2 Context-Free Grammars and EBNF

In Chapter 18, we saw that lexical tokens are governed by **Regular
Grammars (Type-3)**. However, regular expressions cannot describe nested
or recursively balanced structures, such as arbitrarily nested
parentheses `((1 + (2 * (3))))` or nested conditional blocks.

To specify program syntax, we step up the Chomsky hierarchy to
**Context-Free Grammars (Type-2)**.

#### 19.2.1 Formal Definition of a Context-Free Grammar

A Context-Free Grammar is formally defined as a 4-tuple:

$$G = (V, \Sigma, R, S)$$

Where:

- $V$ is a finite set of **Non-Terminal symbols** (syntactic variables
  representing syntactic structures).
- $\Sigma$ is a finite set of **Terminal symbols** (the atomic lexical
  tokens emitted by our lexer, disjoint from $V$).
- $R$ is a finite set of **Production Rules** mapping a single
  non-terminal to a sequence of non-terminals and terminals:
  $$A \to \alpha \quad \text{where } A \in V \text{ and } \alpha \in (V \cup \Sigma)^*$$
- $S \in V$ is the distinguished **Start Symbol**.

#### 19.2.2 Extended Backus-Naur Form (EBNF) Notation

To define compiler grammars concisely, we use **Extended Backus-Naur
Form (EBNF)** notation:

- `A ::= B C` denotes concatenation (Rule $A$ derives $B$ followed by
  $C$).
- `A ::= B | C` denotes alternation (Rule $A$ derives either $B$ or
  $C$).
- `A ::= (B)?` denotes optionality (Zero or one occurrence of $B$).
- `A ::= (B)*` denotes zero or more repetitions (Kleene closure).
- `A ::= (B)+` denotes one or more repetitions.

#### 19.2.3 The Ambiguity Problem

A grammar is **ambiguous** if a single string of tokens can yield two or
more distinct parse trees. Consider the classic naive arithmetic
grammar:

$$E \to E + E \mid E \times E \mid \text{number}$$

For the input string `1 + 2 * 3`, this grammar permits two valid
derivations:

1.  $(1 + 2) \times 3 = 9$ (Tree grouping $+$ above $\times$).
2.  $1 + (2 \times 3) = 7$ (Tree grouping $\times$ above $+$).

In mathematics and programming, multiplication binds tighter than
addition. A compiler grammar must be completely **unambiguous**,
explicitly baking in:

1.  **Operator Precedence:** Higher precedence operators appear deeper
    in the grammar hierarchy.
2.  **Associativity:** Left-associative operators (e.g., subtraction:
    $10 - 4 - 2 \equiv (10 - 4) - 2$) grow trees downward to the left;
    right-associative operators (e.g., exponentiation:
    $2^{3^2} \equiv 2^{(3^2)}$ or assignment: $a = b = c$) grow downward
    to the right.

### 19.3 Parsing Paradigms: Top-Down vs Bottom-Up

Compiler front-ends historically fall into two major architectural
paradigms (see Table 19–1)

| Top-Down Parsers (LL)           | Bottom-Up Parsers (LR)           |
|:--------------------------------|:---------------------------------|
| Starts at Start Symbol S        | Starts at leaf tokens            |
| Predicts productions downward   | Shifts tokens into a stack       |
| LL(1), LL(k), Recursive Descent | Reduces patterns into non-terms  |
| Hand-crafted, pristine errors   | LR(1), LALR(1) (Yacc, Bison)     |
| Used by: Clang, Rust, V8, tsc   | Extremely rigid, poor error msgs |

Table 19–1: Parsing Paradigms {#tbl:parsing-paradigms}

#### 19.3.1 Why Production Compilers Choose Hand-Crafted Recursive Descent

While table-driven LALR(1) parser generators (such as Yacc and Bison)
were popular in the 1970s and 1980s, virtually every modern industrial
compiler (**Clang/LLVM**, **rustc**, **Go gc**, **TypeScript tsc**,
**V8**, **Swift**) has abandoned parser generators in favour of
**hand-crafted Recursive Descent**.

The reasons are pragmatic:

1.  **State-of-the-Art Error Diagnostics:** When a syntax error occurs,
    a recursive descent parser has the entire operating system call
    stack matching the grammar hierarchy. It knows precisely which
    expression or statement was expected and can emit insightful
    contextual diagnostics with pinpoint recovery.
2.  **Custom Disambiguation:** Complex contextual edge cases (e.g., C++
    template brackets `vector<vector<int>>` vs right-shift operators
    `>>`) are trivial to disambiguate procedurally with arbitrary
    lookahead, but cause shift-reduce conflicts in grammar tables.
3.  **Performance:** Direct C++ function calls with predictable control
    flow execute substantially faster on modern CPU branch predictors
    than table-driven state transitions.

### 19.4 Pratt Parsing: Top-Down Operator Precedence

Standard recursive descent handles structured statements cleanly, but
when parsing deeply nested binary expressions with 10+ levels of
operator precedence, a naive grammar requires 10+ distinct recursive
functions (`parse_expr` $\to$ `parse_equality` $\to$ `parse_relational`
$\to$ `parse_additive` $\to$ `parse_multiplicative` $\dots$). This
introduces call stack overhead and makes adding new operators tedious.

In 1973, **Vaughan Pratt** introduced **Top-Down Operator Precedence
(Pratt Parsing)**. Pratt's insight was to associate two numerical values
with every token:

1.  **Prefix Denotation (`nud` - Null Denotation):** How a token behaves
    at the *start* of an expression (e.g., integer literals,
    identifiers, unary `-`, unary `!`, grouping `(`).
2.  **Infix Denotation (`led` - Left Denotation):** How a token behaves
    *between* two expressions (e.g., `+`, `*`, `==`, function calls `(`,
    index access `[`).
3.  **Binding Power (Precedence):** An integer defining how tightly the
    operator binds its left and right operands.

<figure>
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA4MDAgMjQwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyNDAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0iYm94U2hhZG93IiB4PSItNSUiIHk9Ii01
JSIgd2lkdGg9IjExMCUiIGhlaWdodD0iMTIwJSI+CiAgICAgIDxmZURyb3BTaGFkb3cgZHg9IjAi
IGR5PSIyIiBzdGREZXZpYXRpb249IjIiIGZsb29kLWNvbG9yPSIjMDAwIiBmbG9vZC1vcGFjaXR5
PSIwLjUiLz4KICAgIDwvZmlsdGVyPgogIDwvZGVmcz4KICA8IS0tIFRva2VucyB3aXRoIEJpbmRp
bmcgUG93ZXIgQmFkZ2VzIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDYwLCA0MCkiPgog
ICAgPHRleHQgeD0iMzgwIiB5PSIwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE2
IiBmb250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+UHJhdHQgQmluZGluZyBQ
b3dlciBSZXNvbHV0aW9uIGZvcjogMSArIDIgKiAzID09IDc8L3RleHQ+CiAgICA8IS0tIFRva2Vu
IDE6ICIxIiAtLT4KICAgIDxyZWN0IHg9IjAiIHk9IjMwIiB3aWR0aD0iNzAiIGhlaWdodD0iNTAi
IHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNr
KSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiIGZpbHRlcj0idXJs
KCNib3hTaGFkb3cpIi8+CiAgICA8dGV4dCB4PSIzNSIgeT0iNjAiIGZpbGw9InZhcigtLXN2Zy1m
ZykiIGZvbnQtc2l6ZT0iMTYiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxl
Ij4xPC90ZXh0PgogICAgPCEtLSBUb2tlbiAyOiAiKyIgLS0+CiAgICA8cmVjdCB4PSIxMTAiIHk9
IjMwIiB3aWR0aD0iODAiIGhlaWdodD0iNTAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0x
LCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ry
b2tlLXdpZHRoPSIyIiBmaWx0ZXI9InVybCgjYm94U2hhZG93KSIvPgogICAgPHRleHQgeD0iMTUw
IiB5PSI1NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9udC13ZWlnaHQ9
ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPis8L3RleHQ+CiAgICA8cmVjdCB4PSIxMjAiIHk9
IjYyIiB3aWR0aD0iNjAiIGhlaWdodD0iMTUiIHJ4PSIzIiBmaWxsPSJ2YXIoLS1zdmctdGludC0x
LCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiLz4KICAgIDx0ZXh0IHg9IjE1MCIgeT0iNzMi
IGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRkbGUi
PkJQOiA0MDwvdGV4dD4KICAgIDwhLS0gVG9rZW4gMzogIjIiIC0tPgogICAgPHJlY3QgeD0iMjMw
IiB5PSIzMCIgd2lkdGg9IjcwIiBoZWlnaHQ9IjUwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRp
bnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZyki
IHN0cm9rZS13aWR0aD0iMS41IiBmaWx0ZXI9InVybCgjYm94U2hhZG93KSIvPgogICAgPHRleHQg
eD0iMjY1IiB5PSI2MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9udC13
ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPjI8L3RleHQ+CiAgICA8IS0tIFRva2Vu
IDQ6ICIqIiAtLT4KICAgIDxyZWN0IHg9IjM0MCIgeT0iMzAiIHdpZHRoPSI4MCIgaGVpZ2h0PSI1
MCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJh
Y2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGZpbHRlcj0idXJs
KCNib3hTaGFkb3cpIi8+CiAgICA8dGV4dCB4PSIzODAiIHk9IjU1IiBmaWxsPSJ2YXIoLS1zdmct
ZmcpIiBmb250LXNpemU9IjE2IiBmb250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRs
ZSI+KjwvdGV4dD4KICAgIDxyZWN0IHg9IjM1MCIgeT0iNjIiIHdpZHRoPSI2MCIgaGVpZ2h0PSIx
NSIgcng9IjMiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJh
Y2spKSIvPgogICAgPHRleHQgeD0iMzgwIiB5PSI3MyIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9u
dC1zaXplPSIxNCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+QlA6IDUwPC90ZXh0PgogICAgPCEtLSBU
b2tlbiA1OiAiMyIgLS0+CiAgICA8cmVjdCB4PSI0NjAiIHk9IjMwIiB3aWR0aD0iNzAiIGhlaWdo
dD0iNTAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZh
bGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiIGZpbHRl
cj0idXJsKCNib3hTaGFkb3cpIi8+CiAgICA8dGV4dCB4PSI0OTUiIHk9IjYwIiBmaWxsPSJ2YXIo
LS1zdmctZmcpIiBmb250LXNpemU9IjE2IiBmb250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9
Im1pZGRsZSI+MzwvdGV4dD4KICAgIDwhLS0gVG9rZW4gNjogIj09IiAtLT4KICAgIDxyZWN0IHg9
IjU3MCIgeT0iMzAiIHdpZHRoPSI4MCIgaGVpZ2h0PSI1MCIgcng9IjYiIGZpbGw9InZhcigtLXN2
Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmct
ZmcpIiBzdHJva2Utd2lkdGg9IjIiIGZpbHRlcj0idXJsKCNib3hTaGFkb3cpIi8+CiAgICA8dGV4
dCB4PSI2MTAiIHk9IjU1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE2IiBmb250
LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+PT08L3RleHQ+CiAgICA8cmVjdCB4
PSI1ODAiIHk9IjYyIiB3aWR0aD0iNjAiIGhlaWdodD0iMTUiIHJ4PSIzIiBmaWxsPSJ2YXIoLS1z
dmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiLz4KICAgIDx0ZXh0IHg9IjYx
MCIgeT0iNzMiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9y
PSJtaWRkbGUiPkJQOiAyMDwvdGV4dD4KICAgIDwhLS0gVG9rZW4gNzogIjciIC0tPgogICAgPHJl
Y3QgeD0iNjkwIiB5PSIzMCIgd2lkdGg9IjcwIiBoZWlnaHQ9IjUwIiByeD0iNiIgZmlsbD0idmFy
KC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigt
LXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMS41IiBmaWx0ZXI9InVybCgjYm94U2hhZG93KSIvPgog
ICAgPHRleHQgeD0iNzI1IiB5PSI2MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIx
NiIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPjc8L3RleHQ+CiAgPC9n
PgogIDwhLS0gRXhwbGFuYXRpb24gY2FsbG91dHMgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xh
dGUoNjAsIDE1MCkiPgogICAgPHJlY3QgeD0iMCIgeT0iMCIgd2lkdGg9Ijc2MCIgaGVpZ2h0PSIx
MDUiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxi
YWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHRleHQgeD0iMjAiIHk9IjI1IiBm
aWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCI+QWxn
b3JpdGhtIEZsb3c6PC90ZXh0PgogICAgPHRleHQgeD0iMjAiIHk9IjQ1IiBmaWxsPSJ2YXIoLS1z
dmctZmcpIiBmb250LXNpemU9IjE0Ij4KICAgICAgMS4gUGFyc2VyIGVudGVycyB3aXRoIG1pbl9i
cCA9IDAuIENvbnN1bWVzICcxJy4gU2VlcyAnKycgd2l0aCBCUCA0MCAmZ3Q7IDAuCiAgICA8L3Rl
eHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iNjAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6
ZT0iMTQiPgogICAgICAyLiBSZWN1cnNpdmVseSBwYXJzZXMgUkhTIG9mICcrJyB3aXRoIG1pbl9i
cCA9IDQwLiBDb25zdW1lcyAnMicuIFNlZXMgJyonIHdpdGggQlAgNTAgJmd0OyA0MC4KICAgIDwv
dGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSI3NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1z
aXplPSIxNCI+CiAgICAgIDMuIFJlY3Vyc2l2ZWx5IHBhcnNlcyBSSFMgb2YgJyonIHdpdGggbWlu
X2JwID0gNTAuIENvbnN1bWVzICczJy4gU2VlcyAnPT0nIHdpdGggQlAgMjAgJmx0OyA1MC4KICAg
IDwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSI5MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9u
dC1zaXplPSIxNCI+CiAgICAgICYjeGEwOyYjeGEwOyYjeGEwOyhSZXR1cm5zIGltbWVkaWF0ZWx5
ISkKICAgIDwvdGV4dD4KICA8L2c+Cjwvc3ZnPg==
" alt="Binding Power Resolution" />
<figcaption>Binding Power Resolution</figcaption>
</figure>

Pratt parsing collapses dozens of recursive grammar levels into a single
elegant loop:

``` cpp
std::unique_ptr<ExprAST> parse_expression(int min_precedence) {
    auto left = parse_prefix(); // nud: Literal, Unary, Identifier, Grouping

    while (current_token_precedence() > min_precedence) {
        left = parse_infix(std::move(left)); // led: Binary Operator, Call
    }

    return left;
}
```

### 19.5 Complete Grammar Specification for Faberge

We now formalise the complete Context-Free Grammar for **Faberge** in
EBNF:

```
Program        ::= Statement* EOF

Statement      ::= LetBinding
                 | ReturnStmt
                 | ExprStatement

LetBinding     ::= "let" Identifier (":" Type)? "=" Expression ";"
ReturnStmt     ::= "return" Expression ";"
ExprStatement  ::= Expression ";"

Type           ::= "Int" | "Float" | "String" | "Bool"
                 | "(" Type "->" Type ")"
                 | "[" Type "]"

Expression     ::= LogicalOrExpr

LogicalOrExpr  ::= LogicalAndExpr ( "||" LogicalAndExpr )*
LogicalAndExpr ::= EqualityExpr ( "&&" EqualityExpr )*
EqualityExpr   ::= RelationalExpr ( ("==" | "!=") RelationalExpr )*
RelationalExpr ::= AdditiveExpr ( ("<" | "<=" | ">" | ">=") AdditiveExpr )*
AdditiveExpr   ::= MultiplicativeExpr ( ("+" | "-") MultiplicativeExpr )*
MultiplicativeExpr ::= UnaryExpr ( ("*" | "/" | "%") UnaryExpr )*

UnaryExpr      ::= ("-" | "!") UnaryExpr
                 | PostfixExpr

PostfixExpr    ::= PrimaryExpr ( "(" ArgumentList? ")" )*

PrimaryExpr    ::= Literal_Integer
                 | Literal_Float
                 | Literal_String
                 | Literal_Boolean
                 | Identifier
                 | "(" Expression ")"
                 | BlockExpr
                 | IfExpr
                 | LambdaExpr

BlockExpr      ::= "{" Statement* (Expression)? "}"
IfExpr         ::= "if" Expression "then" Expression "else" Expression
LambdaExpr     ::= "fn" "(" ParameterList? ")" (":" Type)? "=>" Expression

ArgumentList   ::= Expression ( "," Expression )*
ParameterList  ::= Parameter ( "," Parameter )*
Parameter      ::= Identifier ":" Type
```

### 19.6 Experimental Implementation in Modern C++: Designing the AST

Next is a design for an object-oriented, polymorphic Abstract Syntax
Tree in Modern C++20.

#### 19.6.1 Key Architectural Decisions:

1.  **Memory Management:** Explicit ownership using
    `std::unique_ptr<Node>`. Subtrees are uniquely owned by their parent
    nodes, guaranteeing leak-free RAII destruction.
2.  **Source Span Tracking:** Every single AST node embeds a
    `SourceLocation`, enabling downstream type checkers (Chapter 20) and
    code emitters (Chapter 21) to pinpoint source locations in error
    diagnostics.
3.  **The Visitor Pattern:** We equip our AST with a double-dispatch
    `ASTVisitor` interface, separating node data structures from
    algorithmic passes (pretty-printing, type checking, and code
    emission).

<figure id="fig:ast-hierarchy">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI2MTRwdCIgaGVpZ2h0PSIxMzJwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCA2MTQuMDAgMTMyLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDEy
OCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gQVNUTm9kZSAtLT4KPGcgaWQ9Im5vZGUxIiBjbGFz
cz0ibm9kZSI+Cjx0aXRsZT5BU1ROb2RlPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgcG9pbnRzPSIyOTQsLTEyNCAyMDgsLTEyNCAyMDgsLTg4IDI5NCwtODggMjk0
LC0xMjQiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjUxIiB5PSItMTA5IiBmb250
LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1z
aXplPSIxMC4wMCI+Jmx0OyZsdDthYnN0cmFjdCZndDsmZ3Q7PC90ZXh0Pgo8dGV4dCB0ZXh0LWFu
Y2hvcj0ibWlkZGxlIiB4PSIyNTEiIHk9Ii05OCIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBE
ZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkFTVE5vZGU8L3Rl
eHQ+CjwvZz4KPCEtLSBTdG10QVNUIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRp
dGxlPlN0bXRBU1Q8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBw
b2ludHM9IjE5MCwtNjYgMTA0LC02NiAxMDQsLTQ0IDE5MCwtNDQgMTkwLC02NiIvPgo8dGV4dCB0
ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxNDciIHk9Ii01Mi41IiBmb250LWZhbWlseT0iQ291cmll
ciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+U3Rt
dEFTVDwvdGV4dD4KPC9nPgo8IS0tIEFTVE5vZGUmIzQ1OyZndDtTdG10QVNUIC0tPgo8ZyBpZD0i
ZWRnZTEiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPkFTVE5vZGUmIzQ1OyZndDtTdG10QVNUPC90aXRs
ZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTIxNC45MiwtODhDMTk5LjQz
LC04MC43IDE4MS44NiwtNzIuNDIgMTY4LjQ3LC02Ni4xMSIvPgo8L2c+CjwhLS0gRXhwckFTVCAt
LT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5FeHByQVNUPC90aXRsZT4KPHBv
bHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzOTgsLTY2IDMxMiwtNjYg
MzEyLC00NCAzOTgsLTQ0IDM5OCwtNjYiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0i
MzU1IiB5PSItNTIuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25v
LCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkV4cHJBU1Q8L3RleHQ+CjwvZz4KPCEtLSBB
U1ROb2RlJiM0NTsmZ3Q7RXhwckFTVCAtLT4KPGcgaWQ9ImVkZ2UyIiBjbGFzcz0iZWRnZSI+Cjx0
aXRsZT5BU1ROb2RlJiM0NTsmZ3Q7RXhwckFTVDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0
cm9rZT0iYmxhY2siIGQ9Ik0yODcuMDgsLTg4QzMwMi41NywtODAuNyAzMjAuMTQsLTcyLjQyIDMz
My41MywtNjYuMTEiLz4KPC9nPgo8IS0tIExldFN0bXRBU1QgLS0+CjxnIGlkPSJub2RlNCIgY2xh
c3M9Im5vZGUiPgo8dGl0bGU+TGV0U3RtdEFTVDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUi
IHN0cm9rZT0iYmxhY2siIHBvaW50cz0iODYsLTIyIDAsLTIyIDAsMCA4NiwwIDg2LC0yMiIvPgo8
dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI0MyIgeT0iLTguNSIgZm9udC1mYW1pbHk9IkNv
dXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAi
PkxldFN0bXRBU1Q8L3RleHQ+CjwvZz4KPCEtLSBTdG10QVNUJiM0NTsmZ3Q7TGV0U3RtdEFTVCAt
LT4KPGcgaWQ9ImVkZ2UzIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5TdG10QVNUJiM0NTsmZ3Q7TGV0
U3RtdEFTVDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xMjIu
MDksLTQzLjk0QzEwNS43NCwtMzcuMzQgODQuNDQsLTI4Ljc0IDY4LjA3LC0yMi4xMiIvPgo8L2c+
CjwhLS0gRXhwclN0bXRBU1QgLS0+CjxnIGlkPSJub2RlNSIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+
RXhwclN0bXRBU1Q8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBw
b2ludHM9IjE5MCwtMjIgMTA0LC0yMiAxMDQsMCAxOTAsMCAxOTAsLTIyIi8+Cjx0ZXh0IHRleHQt
YW5jaG9yPSJtaWRkbGUiIHg9IjE0NyIgeT0iLTguNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3
LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkV4cHJTdG10
QVNUPC90ZXh0Pgo8L2c+CjwhLS0gU3RtdEFTVCYjNDU7Jmd0O0V4cHJTdG10QVNUIC0tPgo8ZyBp
ZD0iZWRnZTQiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPlN0bXRBU1QmIzQ1OyZndDtFeHByU3RtdEFT
VDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xNDcsLTQzLjgz
QzE0NywtMzcuMjMgMTQ3LC0yOC42NiAxNDcsLTIyLjA3Ii8+CjwvZz4KPCEtLSBCaW5hcnlFeHBy
IC0tPgo8ZyBpZD0ibm9kZTYiIGNsYXNzPSJub2RlIj4KPHRpdGxlPkJpbmFyeUV4cHI8L3RpdGxl
Pgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjI5NCwtMjIgMjA4
LC0yMiAyMDgsMCAyOTQsMCAyOTQsLTIyIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9
IjI1MSIgeT0iLTguNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25v
LCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkJpbmFyeUV4cHI8L3RleHQ+CjwvZz4KPCEt
LSBFeHByQVNUJiM0NTsmZ3Q7QmluYXJ5RXhwciAtLT4KPGcgaWQ9ImVkZ2U1IiBjbGFzcz0iZWRn
ZSI+Cjx0aXRsZT5FeHByQVNUJiM0NTsmZ3Q7QmluYXJ5RXhwcjwvdGl0bGU+CjxwYXRoIGZpbGw9
Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0zMzAuMDksLTQzLjk0QzMxMy43NCwtMzcuMzQgMjky
LjQ0LC0yOC43NCAyNzYuMDcsLTIyLjEyIi8+CjwvZz4KPCEtLSBJZkV4cHIgLS0+CjxnIGlkPSJu
b2RlNyIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+SWZFeHByPC90aXRsZT4KPHBvbHlnb24gZmlsbD0i
bm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzOTgsLTIyIDMxMiwtMjIgMzEyLDAgMzk4LDAg
Mzk4LC0yMiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIzNTUiIHk9Ii04LjUiIGZv
bnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250
LXNpemU9IjEwLjAwIj5JZkV4cHI8L3RleHQ+CjwvZz4KPCEtLSBFeHByQVNUJiM0NTsmZ3Q7SWZF
eHByIC0tPgo8ZyBpZD0iZWRnZTYiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPkV4cHJBU1QmIzQ1OyZn
dDtJZkV4cHI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMzU1
LC00My44M0MzNTUsLTM3LjIzIDM1NSwtMjguNjYgMzU1LC0yMi4wNyIvPgo8L2c+CjwhLS0gTGFt
YmRhRXhwciAtLT4KPGcgaWQ9Im5vZGU4IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5MYW1iZGFFeHBy
PC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSI1MDIs
LTIyIDQxNiwtMjIgNDE2LDAgNTAyLDAgNTAyLC0yMiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIiB4PSI0NTkiIHk9Ii04LjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNh
bnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5MYW1iZGFFeHByPC90ZXh0Pgo8
L2c+CjwhLS0gRXhwckFTVCYjNDU7Jmd0O0xhbWJkYUV4cHIgLS0+CjxnIGlkPSJlZGdlNyIgY2xh
c3M9ImVkZ2UiPgo8dGl0bGU+RXhwckFTVCYjNDU7Jmd0O0xhbWJkYUV4cHI8L3RpdGxlPgo8cGF0
aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMzc5LjkxLC00My45NEMzOTYuMjYsLTM3
LjM0IDQxNy41NiwtMjguNzQgNDMzLjkzLC0yMi4xMiIvPgo8L2c+CjwhLS0gQ2FsbEV4cHIgLS0+
CjxnIGlkPSJub2RlOSIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+Q2FsbEV4cHI8L3RpdGxlPgo8cG9s
eWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjYwNiwtMjIgNTIwLC0yMiA1
MjAsMCA2MDYsMCA2MDYsLTIyIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjU2MyIg
eT0iLTguNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25v
c3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkNhbGxFeHByPC90ZXh0Pgo8L2c+CjwhLS0gRXhwckFT
VCYjNDU7Jmd0O0NhbGxFeHByIC0tPgo8ZyBpZD0iZWRnZTgiIGNsYXNzPSJlZGdlIj4KPHRpdGxl
PkV4cHJBU1QmIzQ1OyZndDtDYWxsRXhwcjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIGQ9Ik0zOTguMSwtNDUuM0M0MzMuOCwtMzguMDkgNDg0LjI3LC0yNy45IDUxOS45
NiwtMjAuNjkiLz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 19–2: AST Polymorphic Class Hierarchy</figcaption>
</figure>

#### 19.6.2 Implementation

The listings below implement and test the C++ classes needed to complete
the functionality outlined in this chapter:

- Listing 19–1 defines the foundational AST definitions using move
  semantics and binary tree classes.
- Listing 19–2 uses the Visitor Pattern to create a source-file
  pretty-printer for easy side-by-side analysis.
- Listing 19–3 constructs the core parser engine, combining structured
  recursive descent with Pratt precedence climbing and **Panic-Mode
  Error Recovery**.
- Listing 19–4 implements a comprehensive driver to parse an expressive
  functional **Faberge** program, construct the AST, and render the
  resulting tree hierarchy.

Listing 19–1: AST Node Hierarchy (AST.hpp):

In [ ]:
#ifndef AST_HPP
#define AST_HPP

#include "faberge-headers/Token.hpp"
#include <string>
#include <vector>
#include <memory>
#include <string_view>
#include <optional>

namespace Faberge {

// Forward declarations
class ASTVisitor;

// -----------------------------------------------------------------------------
// Base AST Node
// -----------------------------------------------------------------------------
class ASTNode {
public:
    explicit ASTNode(SourceLocation loc) : m_location(loc) {}
    virtual ~ASTNode() = default;

    [[nodiscard]] SourceLocation location() const noexcept { return m_location; }
    virtual void accept(ASTVisitor& visitor) const = 0;

private:
    SourceLocation m_location;
};

// -----------------------------------------------------------------------------
// Types Representation
// -----------------------------------------------------------------------------
enum class PrimitiveTypeKind {
    Int,
    Float,
    String,
    Bool,
    Void
};

class TypeNode {
public:
    enum class Kind { Primitive, Function, List };

    explicit TypeNode(PrimitiveTypeKind prim) 
        : m_kind(Kind::Primitive), m_primitive_kind(prim) {}

    static TypeNode make_function(TypeNode param, TypeNode ret) {
        TypeNode t(Kind::Function);
        t.m_param_types.push_back(std::make_shared<TypeNode>(std::move(param)));
        t.m_return_type = std::make_shared<TypeNode>(std::move(ret));
        return t;
    }

    [[nodiscard]] Kind kind() const noexcept { return m_kind; }
    [[nodiscard]] PrimitiveTypeKind primitive_kind() const noexcept { return m_primitive_kind; }

    [[nodiscard]] std::string to_string() const {
        switch (m_kind) {
            case Kind::Primitive:
                switch (m_primitive_kind) {
                    case PrimitiveTypeKind::Int:    return "Int";
                    case PrimitiveTypeKind::Float:  return "Float";
                    case PrimitiveTypeKind::String: return "String";
                    case PrimitiveTypeKind::Bool:   return "Bool";
                    case PrimitiveTypeKind::Void:   return "Void";
                }
                break;
            case Kind::Function:
                return "(" + (m_param_types.empty() ? "Void" : m_param_types[0]->to_string()) + 
                       " -> " + m_return_type->to_string() + ")";
            case Kind::List:
                return "[" + m_param_types[0]->to_string() + "]";
        }
        return "Unknown";
    }

private:
    explicit TypeNode(Kind k) : m_kind(k) {}
    Kind m_kind;
    PrimitiveTypeKind m_primitive_kind{PrimitiveTypeKind::Void};
    std::vector<std::shared_ptr<TypeNode>> m_param_types;
    std::shared_ptr<TypeNode> m_return_type;
};

// -----------------------------------------------------------------------------
// Expression & Statement Base Nodes (Defined before any concrete AST nodes)
// -----------------------------------------------------------------------------

class ExprAST : public ASTNode {
public:
    using ASTNode::ASTNode;
};

class StmtAST : public ASTNode {
public:
    using ASTNode::ASTNode;
};

// -----------------------------------------------------------------------------
// Expression Nodes
// -----------------------------------------------------------------------------

class LiteralExprAST : public ExprAST {
public:
    LiteralExprAST(SourceLocation loc, TokenType type, std::string_view value)
        : ExprAST(loc), m_token_type(type), m_value(value) {}

    [[nodiscard]] TokenType token_type() const noexcept { return m_token_type; }
    [[nodiscard]] std::string_view value() const noexcept { return m_value; }

    void accept(ASTVisitor& visitor) const override;

private:
    TokenType m_token_type;
    std::string m_value;
};

class VariableExprAST : public ExprAST {
public:
    VariableExprAST(SourceLocation loc, std::string name)
        : ExprAST(loc), m_name(std::move(name)) {}

    [[nodiscard]] const std::string& name() const noexcept { return m_name; }

    void accept(ASTVisitor& visitor) const override;

private:
    std::string m_name;
};

class UnaryExprAST : public ExprAST {
public:
    UnaryExprAST(SourceLocation loc, TokenType op, std::unique_ptr<ExprAST> operand)
        : ExprAST(loc), m_op(op), m_operand(std::move(operand)) {}

    [[nodiscard]] TokenType op() const noexcept { return m_op; }
    [[nodiscard]] const ExprAST& operand() const noexcept { return *m_operand; }

    void accept(ASTVisitor& visitor) const override;

private:
    TokenType m_op;
    std::unique_ptr<ExprAST> m_operand;
};

class BinaryExprAST : public ExprAST {
public:
    BinaryExprAST(SourceLocation loc, TokenType op, 
                   std::unique_ptr<ExprAST> left, 
                   std::unique_ptr<ExprAST> right)
        : ExprAST(loc), m_op(op), m_left(std::move(left)), m_right(std::move(right)) {}

    [[nodiscard]] TokenType op() const noexcept { return m_op; }
    [[nodiscard]] const ExprAST& left() const noexcept { return *m_left; }
    [[nodiscard]] const ExprAST& right() const noexcept { return *m_right; }

    void accept(ASTVisitor& visitor) const override;

private:
    TokenType m_op;
    std::unique_ptr<ExprAST> m_left;
    std::unique_ptr<ExprAST> m_right;
};

class IfExprAST : public ExprAST {
public:
    IfExprAST(SourceLocation loc, 
               std::unique_ptr<ExprAST> condition, 
               std::unique_ptr<ExprAST> then_branch, 
               std::unique_ptr<ExprAST> else_branch)
        : ExprAST(loc), m_condition(std::move(condition)), 
          m_then_branch(std::move(then_branch)), m_else_branch(std::move(else_branch)) {}

    [[nodiscard]] const ExprAST& condition() const noexcept { return *m_condition; }
    [[nodiscard]] const ExprAST& then_branch() const noexcept { return *m_then_branch; }
    [[nodiscard]] const ExprAST& else_branch() const noexcept { return *m_else_branch; }

    void accept(ASTVisitor& visitor) const override;

private:
    std::unique_ptr<ExprAST> m_condition;
    std::unique_ptr<ExprAST> m_then_branch;
    std::unique_ptr<ExprAST> m_else_branch;
};

class CallExprAST : public ExprAST {
public:
    CallExprAST(SourceLocation loc, std::unique_ptr<ExprAST> callee, 
                 std::vector<std::unique_ptr<ExprAST>> args)
        : ExprAST(loc), m_callee(std::move(callee)), m_arguments(std::move(args)) {}

    [[nodiscard]] const ExprAST& callee() const noexcept { return *m_callee; }
    [[nodiscard]] const std::vector<std::unique_ptr<ExprAST>>& arguments() const noexcept { 
        return m_arguments; 
    }

    void accept(ASTVisitor& visitor) const override;

private:
    std::unique_ptr<ExprAST> m_callee;
    std::vector<std::unique_ptr<ExprAST>> m_arguments;
};

class BlockExprAST : public ExprAST {
public:
    BlockExprAST(SourceLocation loc, 
                 std::vector<std::unique_ptr<StmtAST>> statements,
                 std::unique_ptr<ExprAST> result_expr) : ExprAST(loc), m_statements(std::move(statements)), m_result_expr(std::move(result_expr)) {}
    [[nodiscard]] const std::vector<std::unique_ptr<StmtAST>>& statements() const noexcept { 
        return m_statements; 
    }
    [[nodiscard]] const ExprAST* result_expr() const noexcept { return m_result_expr.get(); }

    void accept(ASTVisitor& visitor) const override;

private:
    std::vector<std::unique_ptr<StmtAST>> m_statements;
    std::unique_ptr<ExprAST> m_result_expr;
};

struct Parameter {
    std::string name;
    TypeNode type;
    SourceLocation location;
};

class LambdaExprAST : public ExprAST {
public:
    LambdaExprAST(SourceLocation loc, std::vector<Parameter> params, 
                  std::optional<TypeNode> return_type, 
                  std::unique_ptr<ExprAST> body)
        : ExprAST(loc), m_parameters(std::move(params)), 
          m_return_type(std::move(return_type)), m_body(std::move(body)) {}

    [[nodiscard]] const std::vector<Parameter>& parameters() const noexcept { return m_parameters; }
    [[nodiscard]] const std::optional<TypeNode>& return_type() const noexcept { return m_return_type; }
    [[nodiscard]] const ExprAST& body() const noexcept { return *m_body; }

    void accept(ASTVisitor& visitor) const override;

private:
    std::vector<Parameter> m_parameters;
    std::optional<TypeNode> m_return_type;
    std::unique_ptr<ExprAST> m_body;
};

// -----------------------------------------------------------------------------
// Statement Nodes
// -----------------------------------------------------------------------------

class LetStmtAST : public StmtAST {
public:
    LetStmtAST(SourceLocation loc, std::string name, 
               std::optional<TypeNode> type_annot, 
               std::unique_ptr<ExprAST> initialiser)
        : StmtAST(loc), m_name(std::move(name)), 
          m_type_annot(std::move(type_annot)), m_initialiser(std::move(initialiser)) {}

    [[nodiscard]] const std::string& name() const noexcept { return m_name; }
    [[nodiscard]] const std::optional<TypeNode>& type_annotation() const noexcept { return m_type_annot; }
    [[nodiscard]] const ExprAST& initialiser() const noexcept { return *m_initialiser; }

    void accept(ASTVisitor& visitor) const override;

private:
    std::string m_name;
    std::optional<TypeNode> m_type_annot;
    std::unique_ptr<ExprAST> m_initialiser;
};

class ExprStmtAST : public StmtAST {
public:
    ExprStmtAST(SourceLocation loc, std::unique_ptr<ExprAST> expr)
        : StmtAST(loc), m_expression(std::move(expr)) {}

    [[nodiscard]] const ExprAST& expression() const noexcept { return *m_expression; }

    void accept(ASTVisitor& visitor) const override;

private:
    std::unique_ptr<ExprAST> m_expression;
};

// -----------------------------------------------------------------------------
// Program Root Node
// -----------------------------------------------------------------------------
class ProgramAST : public ASTNode {
public:
    ProgramAST(SourceLocation loc, std::vector<std::unique_ptr<StmtAST>> stmts)
        : ASTNode(loc), m_statements(std::move(stmts)) {}

    [[nodiscard]] const std::vector<std::unique_ptr<StmtAST>>& statements() const noexcept { 
        return m_statements; 
    }

    void accept(ASTVisitor& visitor) const override;

private:
    std::vector<std::unique_ptr<StmtAST>> m_statements;
};

} // namespace Faberge

#endif // AST_HPP

Listing 19–2: AST Inspector and Pretty-Printer (ASTVisitor.hpp):

In [ ]:
#ifndef AST_VISTOR_HPP
#define AST_VISTOR_HPP

#include "faberge-headers/AST.hpp"
#include <iostream>

namespace Faberge {

class ASTVisitor {
public:
    virtual ~ASTVisitor() = default;

    virtual void visit(const ProgramAST& node) = 0;
    virtual void visit(const LetStmtAST& node) = 0;
    virtual void visit(const ExprStmtAST& node) = 0;
    virtual void visit(const LiteralExprAST& node) = 0;
    virtual void visit(const VariableExprAST& node) = 0;
    virtual void visit(const UnaryExprAST& node) = 0;
    virtual void visit(const BinaryExprAST& node) = 0;
    virtual void visit(const IfExprAST& node) = 0;
    virtual void visit(const CallExprAST& node) = 0;
    virtual void visit(const BlockExprAST& node) = 0;
    virtual void visit(const LambdaExprAST& node) = 0;
};

// Inline definitions of double-dispatch accept methods
inline void ProgramAST::accept(ASTVisitor& v) const     { v.visit(*this); }
inline void LetStmtAST::accept(ASTVisitor& v) const     { v.visit(*this); }
inline void ExprStmtAST::accept(ASTVisitor& v) const    { v.visit(*this); }
inline void LiteralExprAST::accept(ASTVisitor& v) const { v.visit(*this); }
inline void VariableExprAST::accept(ASTVisitor& v) const{ v.visit(*this); }
inline void UnaryExprAST::accept(ASTVisitor& v) const   { v.visit(*this); }
inline void BinaryExprAST::accept(ASTVisitor& v) const  { v.visit(*this); }
inline void IfExprAST::accept(ASTVisitor& v) const      { v.visit(*this); }
inline void CallExprAST::accept(ASTVisitor& v) const    { v.visit(*this); }
inline void BlockExprAST::accept(ASTVisitor& v) const   { v.visit(*this); }
inline void LambdaExprAST::accept(ASTVisitor& v) const  { v.visit(*this); }

// Diagnostic AST Pretty-Printer
class ASTPrinter : public ASTVisitor {
public:
    void print(const ASTNode& node) {
        m_indent_level = 0;
        node.accept(*this);
    }

    void visit(const ProgramAST& node) override {
        print_indent();
        std::cout << "\033[1;36mProgram\033[0m\n";
        indent();
        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }
        dedent();
    }

    void visit(const LetStmtAST& node) override {
        print_indent();
        std::cout << "\033[1;32mLetStmt\033[0m (name: '\033[1m" << node.name() << "\033[0m'";
        if (node.type_annotation()) {
            std::cout << ", type: \033[33m" << node.type_annotation()->to_string() << "\033[0m";
        }
        std::cout << ")\n";
        indent();
        node.initialiser().accept(*this);
        dedent();
    }

    void visit(const ExprStmtAST& node) override {
        print_indent();
        std::cout << "\033[1;34mExprStmt\033[0m\n";
        indent();
        node.expression().accept(*this);
        dedent();
    }

    void visit(const LiteralExprAST& node) override {
        print_indent();
        std::cout << "\033[33mLiteral\033[0m (" << node.value() << ")\n";
    }

    void visit(const VariableExprAST& node) override {
        print_indent();
        std::cout << "\033[35mVariable\033[0m (" << node.name() << ")\n";
    }

    void visit(const UnaryExprAST& node) override {
        print_indent();
        std::cout << "\033[31mUnaryExpr\033[0m (op: " << Token::to_string(node.op()) << ")\n";
        indent();
        node.operand().accept(*this);
        dedent();
    }

    void visit(const BinaryExprAST& node) override {
        print_indent();
        std::cout << "\033[1;31mBinaryExpr\033[0m (op: " << Token::to_string(node.op()) << ")\n";
        indent();
        node.left().accept(*this);
        node.right().accept(*this);
        dedent();
    }

    void visit(const IfExprAST& node) override {
        print_indent();
        std::cout << "\033[1;35mIfExpr\033[0m\n";
        indent();
        node.condition().accept(*this);
        node.then_branch().accept(*this);
        node.else_branch().accept(*this);
        dedent();
    }

    void visit(const CallExprAST& node) override {
        print_indent();
        std::cout << "\033[1;34mCallExpr\033[0m\n";
        indent();
        node.callee().accept(*this);
        for (const auto& arg : node.arguments()) {
            arg->accept(*this);
        }
        dedent();
    }

    void visit(const BlockExprAST& node) override {
        print_indent();
        std::cout << "\033[1;36mBlockExpr\033[0m\n";
        indent();
        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }
        if (node.result_expr()) {
            node.result_expr()->accept(*this);
        }
        dedent();
    }

    void visit(const LambdaExprAST& node) override {
        print_indent();
        std::cout << "\033[1;32mLambdaExpr\033[0m (";
        for (size_t i = 0; i < node.parameters().size(); ++i) {
            std::cout << node.parameters()[i].name << ": " << node.parameters()[i].type.to_string();
            if (i + 1 < node.parameters().size()) std::cout << ", ";
        }
        std::cout << ")\n";
        indent();
        node.body().accept(*this);
        dedent();
    }

private:
    int m_indent_level{0};

    void indent() noexcept { ++m_indent_level; }
    void dedent() noexcept { --m_indent_level; }

    void print_indent() const {
        for (int i = 0; i < m_indent_level; ++i) {
            std::cout << "  ";
        }
    }
};

} // namespace Faberge

#endif // AST_VISITOR_HPP

Listing 19–3: Recursive Descent and Pratt Parser (Parser.hpp):

In [ ]:
#ifndef PARSER_HPP
#define PARSER_HPP

#include "faberge-headers/Token.hpp"
#include "faberge-headers/AST.hpp"
#include "faberge-headers/Diagnostic.hpp"
#include <vector>
#include <memory>
#include <stdexcept>
#include <span>

namespace Faberge {

class Parser {
public:
    explicit Parser(std::vector<Token> tokens, std::string_view source_code)
        : m_tokens(std::move(tokens)), m_source_code(source_code), m_cursor(0) {}

    std::unique_ptr<ProgramAST> parse_program() {
        std::vector<std::unique_ptr<StmtAST>> statements;
        SourceLocation start_loc = current_token().location;

        while (!is_at_end()) {
            try {
                statements.push_back(parse_statement());
            } catch (const std::runtime_error& e) {
                synchronise();
            }
        }

        return std::make_unique<ProgramAST>(start_loc, std::move(statements));
    }

    [[nodiscard]] bool has_errors() const noexcept { return m_error_count > 0; }
    [[nodiscard]] size_t error_count() const noexcept { return m_error_count; }

private:
    std::vector<Token> m_tokens;
    std::string_view m_source_code;
    size_t m_cursor{0};
    size_t m_error_count{0};

    // -------------------------------------------------------------------------
    // Token Navigation Helpers
    // -------------------------------------------------------------------------
    [[nodiscard]] bool is_at_end() const noexcept {
        return m_cursor >= m_tokens.size() || current_token().type == TokenType::EndOfFile;
    }

    [[nodiscard]] const Token& current_token() const noexcept {
        return m_tokens[m_cursor];
    }

    [[nodiscard]] const Token& previous_token() const noexcept {
        return m_tokens[m_cursor - 1];
    }

    const Token& advance() noexcept {
        if (!is_at_end()) ++m_cursor;
        return previous_token();
    }

    bool check(TokenType type) const noexcept {
        if (is_at_end()) return false;
        return current_token().type == type;
    }

    bool match(TokenType type) noexcept {
        if (check(type)) {
            advance();
            return true;
        }
        return false;
    }

    const Token& consume(TokenType expected, std::string_view err_msg) {
        if (check(expected)) return advance();
        
        ++m_error_count;
        DiagnosticEngine::report_error(m_source_code, current_token().location, err_msg);
        throw std::runtime_error(std::string(err_msg));
    }

    // -------------------------------------------------------------------------
    // Panic-Mode Error Recovery
    // -------------------------------------------------------------------------
    void synchronise() noexcept {
        advance();
        while (!is_at_end()) {
            if (previous_token().type == TokenType::Semicolon) return;

            switch (current_token().type) {
                case TokenType::Keyword_Let:
                case TokenType::Keyword_Fn:
                case TokenType::Keyword_If:
                case TokenType::Keyword_Return:
                    return;
                default:
                    advance();
                    break;
            }
        }
    }

    // -------------------------------------------------------------------------
    // Statement Parsing
    // -------------------------------------------------------------------------
    std::unique_ptr<StmtAST> parse_statement() {
        if (match(TokenType::Keyword_Let)) {
            return parse_let_statement();
        }
        return parse_expression_statement();
    }

    std::unique_ptr<StmtAST> parse_let_statement() {
        SourceLocation loc = previous_token().location;
        Token name_tok = consume(TokenType::Identifier, "Expected variable name after 'let'.");

        std::optional<TypeNode> type_annot;
        if (match(TokenType::Colon)) {
            type_annot = parse_type();
        }

        consume(TokenType::Equal, "Expected '=' in variable declaration.");
        auto init_expr = parse_expression();
        consume(TokenType::Semicolon, "Expected ';' after let statement.");

        return std::make_unique<LetStmtAST>(loc, std::string(name_tok.lexeme), 
                                            std::move(type_annot), std::move(init_expr));
    }

    std::unique_ptr<StmtAST> parse_expression_statement() {
        SourceLocation loc = current_token().location;
        auto expr = parse_expression();
        consume(TokenType::Semicolon, "Expected ';' after expression statement.");
        return std::make_unique<ExprStmtAST>(loc, std::move(expr));
    }

    TypeNode parse_type() {
        if (match(TokenType::Identifier)) {
            std::string_view name = previous_token().lexeme;
            if (name == "Int")    return TypeNode(PrimitiveTypeKind::Int);
            if (name == "Float")  return TypeNode(PrimitiveTypeKind::Float);
            if (name == "String") return TypeNode(PrimitiveTypeKind::String);
            if (name == "Bool")   return TypeNode(PrimitiveTypeKind::Bool);
            if (name == "Void")   return TypeNode(PrimitiveTypeKind::Void);
            throw std::runtime_error("Unknown type identifier '" + std::string(name) + "'.");
        }
        throw std::runtime_error("Expected type name.");
    }

    // -------------------------------------------------------------------------
    // Pratt Expression Parsing
    // -------------------------------------------------------------------------
    static int get_infix_precedence(TokenType type) noexcept {
        switch (type) {
            case TokenType::PipePipe:           return 10; // ||
            case TokenType::AmpersandAmpersand: return 20; // &&
            case TokenType::EqualEqual:
            case TokenType::ExclamationEqual:   return 30; // ==, !=
            case TokenType::Less:
            case TokenType::LessEqual:
            case TokenType::Greater:
            case TokenType::GreaterEqual:       return 40; // <, <=, >, >=
            case TokenType::Plus:
            case TokenType::Minus:              return 50; // +, -
            case TokenType::Asterisk:
            case TokenType::Slash:
            case TokenType::Percent:            return 60; // *, /, %
            case TokenType::LeftParen:          return 70; // Function Call: f(...)
            default:                            return 0;
        }
    }

    std::unique_ptr<ExprAST> parse_expression(int min_precedence = 0) {
        auto left = parse_prefix();

        while (min_precedence < get_infix_precedence(current_token().type)) {
            left = parse_infix(std::move(left));
        }

        return left;
    }

    std::unique_ptr<ExprAST> parse_prefix() {
        SourceLocation loc = current_token().location;

        // 1. Unary Operators: -, !
        if (match(TokenType::Minus) || match(TokenType::Exclamation)) {
            TokenType op = previous_token().type;
            auto operand = parse_expression(60); // High unary binding power
            return std::make_unique<UnaryExprAST>(loc, op, std::move(operand));
        }

        // 2. Literals
        if (match(TokenType::Literal_Integer) || 
            match(TokenType::Literal_Float)   || 
            match(TokenType::Literal_String)  || 
            match(TokenType::Literal_Boolean)) {
            return std::make_unique<LiteralExprAST>(loc, previous_token().type, previous_token().lexeme);
        }

        // 3. Variables and Identifiers
        if (match(TokenType::Identifier)) {
            return std::make_unique<VariableExprAST>(loc, std::string(previous_token().lexeme));
        }

        // 4. Parenthesised Grouping: ( expr )
        if (match(TokenType::LeftParen)) {
            auto inner = parse_expression(0);
            consume(TokenType::RightParen, "Expected ')' after grouping expression.");
            return inner;
        }

        // 5. Block Expressions: { stmt* expr? }
        if (match(TokenType::LeftBrace)) {
            return parse_block_expression();
        }

        // 6. Conditional Expressions: if cond then expr else expr
        if (match(TokenType::Keyword_If)) {
            return parse_if_expression();
        }

        // 7. Lambda Expressions: fn (params) => body
        if (match(TokenType::Keyword_Fn)) {
            return parse_lambda_expression();
        }

        ++m_error_count;
        DiagnosticEngine::report_error(m_source_code, current_token().location, 
                                       "Expected expression, found '" + std::string(current_token().lexeme) + "'.");
        throw std::runtime_error("Syntax error: unexpected expression token.");
    }

    std::unique_ptr<ExprAST> parse_infix(std::unique_ptr<ExprAST> left) {
        SourceLocation loc = current_token().location;
        Token op_tok = advance();

        // Function Call Operator: callee(arg1, arg2)
        if (op_tok.type == TokenType::LeftParen) {
            std::vector<std::unique_ptr<ExprAST>> args;
            if (!check(TokenType::RightParen)) {
                do {
                    args.push_back(parse_expression(0));
                } while (match(TokenType::Comma));
            }
            consume(TokenType::RightParen, "Expected ')' after argument list.");
            return std::make_unique<CallExprAST>(loc, std::move(left), std::move(args));
        }

        // Standard Binary Arithmetic and Logical Operators
        int prec = get_infix_precedence(op_tok.type);
        auto right = parse_expression(prec); // Left-associative recursion

        return std::make_unique<BinaryExprAST>(loc, op_tok.type, std::move(left), std::move(right));
    }

    std::unique_ptr<ExprAST> parse_block_expression() {
        SourceLocation loc = previous_token().location;
        std::vector<std::unique_ptr<StmtAST>> stmts;
        std::unique_ptr<ExprAST> result_expr;

        while (!check(TokenType::RightBrace) && !is_at_end()) {
            if (check(TokenType::Keyword_Let)) {
                stmts.push_back(parse_statement());
            } else {
                // Peek ahead to check if this is an expression-statement (with ';') or a tail expression
                auto expr = parse_expression();
                if (match(TokenType::Semicolon)) {
                    stmts.push_back(std::make_unique<ExprStmtAST>(expr->location(), std::move(expr)));
                } else {
                    result_expr = std::move(expr);
                    break;
                }
            }
        }

        consume(TokenType::RightBrace, "Expected '}' to close block expression.");
        return std::make_unique<BlockExprAST>(loc, std::move(stmts), std::move(result_expr));
    }

    std::unique_ptr<ExprAST> parse_if_expression() {
        SourceLocation loc = previous_token().location;
        auto cond = parse_expression();
        consume(TokenType::Keyword_Then, "Expected 'then' after if condition.");
        auto then_b = parse_expression();
        consume(TokenType::Keyword_Else, "Expected 'else' in conditional expression.");
        auto else_b = parse_expression();

        return std::make_unique<IfExprAST>(loc, std::move(cond), std::move(then_b), std::move(else_b));
    }

    std::unique_ptr<ExprAST> parse_lambda_expression() {
        SourceLocation loc = previous_token().location;
        consume(TokenType::LeftParen, "Expected '(' after 'fn'.");

        std::vector<Parameter> params;
        if (!check(TokenType::RightParen)) {
            do {
                Token param_name = consume(TokenType::Identifier, "Expected parameter name.");
                consume(TokenType::Colon, "Expected ':' after parameter name.");
                TypeNode param_type = parse_type();
                params.push_back({std::string(param_name.lexeme), std::move(param_type), param_name.location});
            } while (match(TokenType::Comma));
        }

        consume(TokenType::RightParen, "Expected ')' after parameter list.");

        std::optional<TypeNode> ret_type;
        if (match(TokenType::Colon)) {
            ret_type = parse_type();
        }

        consume(TokenType::FatArrow, "Expected '=>' in function definition.");
        auto body = parse_expression();

        return std::make_unique<LambdaExprAST>(loc, std::move(params), std::move(ret_type), std::move(body));
    }
};

} // namespace Faberge

#endif // PARSER_HPP

Listing 19–4: Verification and AST Tree Visualisation (main.cpp):

In [ ]:
#include "faberge-headers/Lexer.hpp"
#include "faberge-headers/Parser.hpp"
#include "faberge-headers/ASTVisitor.hpp"
#include <iostream>

void test_parser(std::string_view title, std::string_view source_code) {
    using namespace Faberge;

    std::cout << "\n====================================================\n";
    std::cout << " TEST CASE: " << title << "\n";
    std::cout << "====================================================\n";
    std::cout << "SOURCE CODE:\n" << source_code << "\n----------------------------------------------------\n";

    // 1. Stage 1: Lexical Analysis
    Lexer lexer(source_code, "test.fab");
    auto tokens = lexer.tokenise();

    if (lexer.has_errors()) {
        std::cerr << "[Lexer Error]: Aborting parse due to scan errors.\n";
        return;
    }

    // 2. Stage 2: Syntactic Analysis
    Parser parser(std::move(tokens), source_code);
    auto ast = parser.parse_program();

    if (parser.has_errors()) {
        std::cerr << "\n[Parser Error]: Parsing failed with " << parser.error_count() << " error(s).\n\n";
        return;
    }

    // 3. Render AST
    std::cout << "ABSTRACT SYNTAX TREE (AST):\n";
    ASTPrinter printer;
    printer.print(*ast);
    std::cout << "----------------------------------------------------\n";
    std::cout << "[Success]: AST constructed cleanly.\n\n";
}

int main() {
    // Test 1: Complex Arithmetic and Operator Precedence
    constexpr std::string_view precedence_source = R"(
        let result = 1 + 2 * 3 == 7 && 10 > 5;
    )";
    test_parser("Arithmetic and Precedence Resolution", precedence_source);

    // Test 2: Recursive Functional Pipeline with Lambda and Block Expression
    constexpr std::string_view functional_source = R"(
        let factorial = fn (n: Int): Int => {
            if n <= 1 then 1 else n * factorial(n - 1)
        };

        let total = factorial(5) + 100;
    )";
    test_parser("Higher-Order Functional Pipeline", functional_source);

    // Test 3: Parser Error Diagnostics and Panic-Mode Recovery
    constexpr std::string_view broken_source = R"(
        let valid_a = 10;
        let broken_b = ; // Missing initialiser expression!
        let valid_c = 30 + 40;
    )";
    test_parser("Panic-Mode Error Recovery", broken_source);

    return 0;
}

In [ ]:
main(); // Invocation of main program for Jupyter notebook running the xeus-cpp kernel

### 19.7 Summary

In this second stage of our compiler project, we converted linear token
sequences into structured **Abstract Syntax Trees**:

- **Context-Free Grammars (Type-2)** provide the mathematical framework
  for defining recursively nested syntax.
- **Pratt Parsing (Top-Down Operator Precedence)** combines the
  readability of recursive descent with an $O(1)$ precedence table,
  solving complex operator precedence and associativity without deep
  call stacks.
- **The AST Polymorphic Hierarchy** represents language expressions
  cleanly using Modern C++ `std::unique_ptr` ownership and
  double-dispatch `ASTVisitor` patterns.
- **Panic-Mode Error Recovery** guarantees that compiler front-ends
  synchronise gracefully after syntax errors rather than crashing midway
  through analysis.

### 19.8 Programming Challenges

#### Exercise 19.1: Right-Associative Exponentiation (`^`) in Pratt Parsing

**Context and Theory:**

Binary operators possess two orthogonal grammatical attributes:
**precedence** (which operator binds tighter) and **associativity** (how
operators of equal precedence group together).

Most arithmetic operators are **left-associative**, grouping expressions
from left to right:

$$a - b - c \equiv (a - b) - c$$

The exponentiation operator (`^`), however, is mathematically
**right-associative**, grouping expressions from right to left (see
Figure 19–3):

$$a ^ { b ^ {c} } \equiv a ^ { ( b ^ { c } ) }$$

In a Pratt parser, associativity is governed by the recursive binding
power passed into the right-hand expression call:

- **Left Associativity:** The right-hand operand is parsed with the
  operator's exact precedence: `parse_expression(prec)`. A subsequent
  operator of identical precedence will fail the condition
  `prec < next_prec` and terminate the sub-expression.
- **Right Associativity:** The right-hand operand is parsed with a
  decremented precedence: `parse_expression(prec - 1)`. A subsequent
  operator of identical precedence satisfies the condition and continues
  binding recursively into the right child.

<figure id="fig:left-right-associative">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzNzdwdCIgaGVpZ2h0PSIxNDNwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzNzcuMDAgMTQzLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDEz
OSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gbWludXMxIC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNz
PSJub2RlIj4KPHRpdGxlPm1pbnVzMTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIGN4PSIxMzgiIGN5PSItMTEzIiByeD0iMTEiIHJ5PSIxMSIvPgo8dGV4dCB0ZXh0
LWFuY2hvcj0ibWlkZGxlIiB4PSIxMzgiIHk9Ii0xMTAuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIg
TmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPigmIzQ1
Oyk8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iMCIgeT0iLTEyNyIgZm9udC1m
YW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6
ZT0iMTAuMDAiPkxlZnQmIzQ1O0Fzc29jaWF0aXZlICgmIzQ1Oyk6PC90ZXh0Pgo8L2c+CjwhLS0g
bWludXMyIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRpdGxlPm1pbnVzMjwvdGl0
bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSI3OCIgY3k9Ii02MiIg
cng9IjExIiByeT0iMTEiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNzgiIHk9Ii01
OS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFj
ZSIgZm9udC1zaXplPSIxMC4wMCI+KCYjNDU7KTwvdGV4dD4KPC9nPgo8IS0tIG1pbnVzMSYjNDU7
Jmd0O21pbnVzMiAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5taW51czEm
IzQ1OyZndDttaW51czI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBk
PSJNMTI5LjY2LC0xMDUuMTlDMTIwLjUyLC05Ny43MyAxMDUuNjUsLTg1LjU4IDk0LjIxLC03Ni4y
MyIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSI5Ni4yMywt
NzMuMzcgODYuMjcsLTY5Ljc2IDkxLjgsLTc4Ljc5IDk2LjIzLC03My4zNyIvPgo8L2c+CjwhLS0g
YyAtLT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5jPC90aXRsZT4KPGVsbGlw
c2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjE1OCIgY3k9Ii02MiIgcng9IjExIiBy
eT0iMTEiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTU4IiB5PSItNTkuNSIgZm9u
dC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQt
c2l6ZT0iMTAuMDAiPmM8L3RleHQ+CjwvZz4KPCEtLSBtaW51czEmIzQ1OyZndDtjIC0tPgo8ZyBp
ZD0iZWRnZTIiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPm1pbnVzMSYjNDU7Jmd0O2M8L3RpdGxlPgo8
cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTQxLjg2LC0xMDIuNTVDMTQ0LjI0
LC05Ni43MiAxNDcuMzcsLTg5LjA0IDE1MC4yNSwtODEuOTkiLz4KPHBvbHlnb24gZmlsbD0iYmxh
Y2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTUzLjYyLC04MyAxNTQuMTYsLTcyLjQyIDE0Ny4x
NCwtODAuMzUgMTUzLjYyLC04MyIvPgo8L2c+CjwhLS0gYSAtLT4KPGcgaWQ9Im5vZGU0IiBjbGFz
cz0ibm9kZSI+Cjx0aXRsZT5hPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJi
bGFjayIgY3g9Ijc4IiBjeT0iLTExIiByeD0iMTEiIHJ5PSIxMSIvPgo8dGV4dCB0ZXh0LWFuY2hv
cj0ibWlkZGxlIiB4PSI3OCIgeT0iLTguNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWph
VnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPmE8L3RleHQ+CjwvZz4K
PCEtLSBtaW51czImIzQ1OyZndDthIC0tPgo8ZyBpZD0iZWRnZTMiIGNsYXNzPSJlZGdlIj4KPHRp
dGxlPm1pbnVzMiYjNDU7Jmd0O2E8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBkPSJNNzgsLTUwLjYxQzc4LC00NS4yMSA3OCwtMzguNDIgNzgsLTMyLjA1Ii8+Cjxwb2x5
Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjgxLjUsLTMyLjA0IDc4LC0y
Mi4wNCA3NC41LC0zMi4wNCA4MS41LC0zMi4wNCIvPgo8L2c+CjwhLS0gYiAtLT4KPGcgaWQ9Im5v
ZGU1IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5iPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIg
c3Ryb2tlPSJibGFjayIgY3g9IjE1OCIgY3k9Ii0xMSIgcng9IjExIiByeT0iMTEiLz4KPHRleHQg
dGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTU4IiB5PSItOC41IiBmb250LWZhbWlseT0iQ291cmll
ciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+Yjwv
dGV4dD4KPC9nPgo8IS0tIG1pbnVzMiYjNDU7Jmd0O2IgLS0+CjxnIGlkPSJlZGdlNCIgY2xhc3M9
ImVkZ2UiPgo8dGl0bGU+bWludXMyJiM0NTsmZ3Q7YjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUi
IHN0cm9rZT0iYmxhY2siIGQ9Ik04Ny4xOCwtNTUuMzhDMTAwLjA4LC00Ny40OCAxMjMuODMsLTMy
LjkzIDE0MC4yNSwtMjIuODciLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2si
IHBvaW50cz0iMTQyLjIzLC0yNS43NiAxNDguOTMsLTE3LjU2IDEzOC41NywtMTkuNzkgMTQyLjIz
LC0yNS43NiIvPgo8L2c+CjwhLS0gZXhwMSAtLT4KPGcgaWQ9Im5vZGU2IiBjbGFzcz0ibm9kZSI+
Cjx0aXRsZT5leHAxPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
Y3g9IjI1OCIgY3k9Ii0xMTMiIHJ4PSIxMSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjI1OCIgeT0iLTExMC41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFW
dSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+KF4pPC90ZXh0Pgo8dGV4
dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjEwMiIgeT0iLTk0IiBmb250LWZhbWlseT0iQ291cmll
ciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+ICYj
MTYwO1JpZ2h0JiM0NTtBc3NvY2lhdGl2ZSAoXik6PC90ZXh0Pgo8L2c+CjwhLS0gYTEgLS0+Cjxn
IGlkPSJub2RlNyIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+YTE8L3RpdGxlPgo8ZWxsaXBzZSBmaWxs
PSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMjM4IiBjeT0iLTYyIiByeD0iMTEiIHJ5PSIxMSIv
Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIyMzgiIHk9Ii01OS41IiBmb250LWZhbWls
eT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIx
MC4wMCI+YTwvdGV4dD4KPC9nPgo8IS0tIGV4cDEmIzQ1OyZndDthMSAtLT4KPGcgaWQ9ImVkZ2U1
IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5leHAxJiM0NTsmZ3Q7YTE8L3RpdGxlPgo8cGF0aCBmaWxs
PSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMjU0LjE0LC0xMDIuNTVDMjUxLjc2LC05Ni43MiAy
NDguNjMsLTg5LjA0IDI0NS43NSwtODEuOTkiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9r
ZT0iYmxhY2siIHBvaW50cz0iMjQ4Ljg2LC04MC4zNSAyNDEuODQsLTcyLjQyIDI0Mi4zOCwtODMg
MjQ4Ljg2LC04MC4zNSIvPgo8L2c+CjwhLS0gZXhwMiAtLT4KPGcgaWQ9Im5vZGU4IiBjbGFzcz0i
bm9kZSI+Cjx0aXRsZT5leHAyPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJi
bGFjayIgY3g9IjMxOCIgY3k9Ii02MiIgcng9IjExIiByeT0iMTEiLz4KPHRleHQgdGV4dC1hbmNo
b3I9Im1pZGRsZSIgeD0iMzE4IiB5PSItNTkuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBE
ZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPiheKTwvdGV4dD4K
PC9nPgo8IS0tIGV4cDEmIzQ1OyZndDtleHAyIC0tPgo8ZyBpZD0iZWRnZTYiIGNsYXNzPSJlZGdl
Ij4KPHRpdGxlPmV4cDEmIzQ1OyZndDtleHAyPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgZD0iTTI2Ni4zNCwtMTA1LjE5QzI3NS40OCwtOTcuNzMgMjkwLjM1LC04NS41
OCAzMDEuNzksLTc2LjIzIi8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBw
b2ludHM9IjMwNC4yLC03OC43OSAzMDkuNzMsLTY5Ljc2IDI5OS43NywtNzMuMzcgMzA0LjIsLTc4
Ljc5Ii8+CjwvZz4KPCEtLSBiMSAtLT4KPGcgaWQ9Im5vZGU5IiBjbGFzcz0ibm9kZSI+Cjx0aXRs
ZT5iMTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIyNzgi
IGN5PSItMTEiIHJ4PSIxMSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9
IjI3OCIgeT0iLTguNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25v
LCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPmI8L3RleHQ+CjwvZz4KPCEtLSBleHAyJiM0
NTsmZ3Q7YjEgLS0+CjxnIGlkPSJlZGdlNyIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+ZXhwMiYjNDU7
Jmd0O2IxPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTMxMS4z
OSwtNTIuOTFDMzA1Ljg2LC00Ni4xMyAyOTcuOCwtMzYuMjUgMjkwLjk4LC0yNy45Ii8+Cjxwb2x5
Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjI5My41MiwtMjUuNDcgMjg0
LjQ4LC0xOS45NCAyODguMDksLTI5LjkgMjkzLjUyLC0yNS40NyIvPgo8L2c+CjwhLS0gYzEgLS0+
CjxnIGlkPSJub2RlMTAiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmMxPC90aXRsZT4KPGVsbGlwc2Ug
ZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjM1OCIgY3k9Ii0xMSIgcng9IjExIiByeT0i
MTEiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMzU4IiB5PSItOC41IiBmb250LWZh
bWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXpl
PSIxMC4wMCI+YzwvdGV4dD4KPC9nPgo8IS0tIGV4cDImIzQ1OyZndDtjMSAtLT4KPGcgaWQ9ImVk
Z2U4IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5leHAyJiM0NTsmZ3Q7YzE8L3RpdGxlPgo8cGF0aCBm
aWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMzI0LjYxLC01Mi45MUMzMzAuMTQsLTQ2LjEz
IDMzOC4yLC0zNi4yNSAzNDUuMDIsLTI3LjkiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9r
ZT0iYmxhY2siIHBvaW50cz0iMzQ3LjkxLC0yOS45IDM1MS41MiwtMTkuOTQgMzQyLjQ4LC0yNS40
NyAzNDcuOTEsLTI5LjkiLz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 19–3: Left and Right Associativity</figcaption>
</figure>

**Specification:**

Extend Faberge's Pratt expression parser to support the exponentiation
operator `TokenType::Caret` (`^`). Assign it higher precedence than
multiplication/division ($70$). Implement right-associative recursion in
`parse_infix()`, and update AST nodes to construct the proper binary
expression hierarchy.

``` cpp
class PrattParserExtension {
public:
    static int get_infix_precedence(TokenType type) noexcept {
        switch (type) {
            case TokenType::Caret:    return 70; // Exponentiation: right-associative
            case TokenType::Asterisk:
            case TokenType::Slash:    return 60; // Multiplicative
            case TokenType::Plus:
            case TokenType::Minus:    return 50; // Additive
            default:                  return 0;
        }
    }
    static bool is_right_associative(TokenType type) noexcept {
        return type == TokenType::Caret;
    }
    std::unique_ptr<ExprAST> parse_infix_operator(std::unique_ptr<ExprAST> left, Token op_tok) {
        int prec = get_infix_precedence(op_tok.type);
        int next_min_prec = is_right_associative(op_tok.type) ? (prec - 1) : prec;
        auto right = parse_expression(next_min_prec);
        return std::make_unique<BinaryExprAST>(op_tok.location, op_tok.type, std::move(left), std::move(right));
    }
};
```

**Verification Tasks:**

1.  **Right-Associativity Grouping:** Parse the expression `2 ^ 3 ^ 2`.
    Verify that the root AST node is a `BinaryExprAST(^)` whose left
    child is `Literal(2)` and whose right child is
    `BinaryExprAST(3 ^ 2)` (evaluating to $2^9 = 512$, not $8^2 = 64$).
2.  **Precedence over Multiplication:** Parse `2 * 3 ^ 2`. Verify that
    the root is `BinaryExprAST(*)` with left child `2` and right child
    `BinaryExprAST(3 ^ 2)`.
3.  **Parenthesised Override:** Parse `(2 ^ 3) ^ 2`. Verify that the
    grouping overrides right-associativity, producing a root with left
    child `BinaryExprAST(2 ^ 3)` and right child `Literal(2)`.

#### Exercise 19.2: Subscript Indexing and Array/List Literal AST Nodes

**Context and Theory:**

A functional language requires syntax for constructing sequential data
structures and indexing into them. In Faberge, array/list literals and
subscript index access are specified via:

$$\text{ListLiteral} \to \text{"["} \; (\text{Expr} \; (\text{","} \; \text{Expr})^*)^? \; \text{"]"}$$

$$\text{IndexAccess} \to \text{Expr} \; \text{"["} \; \text{Expr} \; \text{"]"}$$

In Pratt parsing, brackets serve two distinct semantic functions
depending on their syntactic context (see Figure 19–4):

1.  **Prefix Denotation (`nud`):** When `[` appears at the start of an
    expression, it acts as a list constructor literal, parsing
    comma-separated elements until `]`.
2.  **Infix Denotation (`led`):** When `[` appears immediately following
    an expression, it acts as a postfix/infix operator of high
    precedence, parsing the index expression and consuming the closing
    `]`.

<figure id="fig:subscript">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI3MTcuNjAwMDQi
IGhlaWdodD0iNjYuNCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA3MTcuNjAwMDQgNjYu
NCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IENvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAx
M3B4Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0
cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9
Cgouc3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQg
ewogIGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9m
aWxsIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJv
dyB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9k
aWFtb25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9i
IC5lbmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3Zn
Ym9iIC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsK
fQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNv
cGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFy
a2VyLXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdf
b3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwo
I2JpZ19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJh
cnJvdyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0i
NyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxw
b2x5Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4K
ICAgIDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVm
WT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0
LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwv
cG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtl
ciBpZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwv
Y2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZp
ZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJI
ZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0i
NCIgY3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+
CiAgPC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI3
MTcuNjAwMDQiIGhlaWdodD0iNjYuNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjE5Ni45NTAwMSIgeT0iMTEuNzAwMDAxIiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSIyMTIuNTUiIHk9IjExLjcwMDAwMSIgPjEsPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIzNS45NTAwMSIgeT0iMTEuNzAwMDAxIiA+Miw8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjU5LjM1IiB5PSIxMS43MDAwMDEi
ID4zPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI3NC45NSIgeT0iMTEu
NzAwMDAxIiA+XTwvdGV4dD4KICA8bGluZSB4MT0iMzI3LjYiIHkxPSI3LjgiIHgyPSIzNTEiIHky
PSI3LjgiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzUxLDMuOSAz
NTguODAwMDIsNy44IDM1MSwxMS43MDAwMDEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzNjguNTUwMDIiIHk9IjExLjcwMDAwMSIgPkxp
c3RMaXRlcmFsRXhwckFTVDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIx
OTYuOTUwMDEiIHk9IjQyLjkiID5tYXRyaXhbcm93XVtjb2xdPC90ZXh0PgogIDxsaW5lIHgxPSIz
MjcuNiIgeTE9IjM5IiB4Mj0iMzUxIiB5Mj0iMzkiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxw
b2x5Z29uIHBvaW50cz0iMzUxLDM1LjEwMDAwMiAzNTguODAwMDIsMzkgMzUxLDQyLjkiIGNsYXNz
PSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjk1
IiB5PSIxMS43MDAwMDEiID5BcnJheSBMaXRlcmFsIChQcmVmaXgpOjwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjk1IiB5PSI0Mi45IiA+QXJyYXkgSW5kZXggKEluZml4
KTo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzY4LjU1MDAyIiB5PSI0
Mi45IiA+SW5kZXhFeHByQVNUKEluZGV4RXhwckFTVChtYXRyaXgsIHJvdyksIGNvbCk8L3RleHQ+
Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 19–4: Prefix and Infix Denotation</figcaption>
</figure>

**Specification:**

Create `ListLiteralExprAST` and `IndexExprAST` classes inheriting from
`ExprAST`. Extend `parse_prefix()` to parse list literals upon
encountering `TokenType::LeftBracket`, and extend `parse_infix()` to
parse array subscripting with high precedence ($80$).

``` cpp
class ListLiteralExprAST : public ExprAST {
public:
    ListLiteralExprAST(SourceLocation loc, std::vector<std::unique_ptr<ExprAST>> elements)
        : ExprAST(loc), m_elements(std::move(elements)) {}
    const auto& elements() const noexcept { return m_elements; }
    void accept(ASTVisitor& visitor) const override;
private:
    std::vector<std::unique_ptr<ExprAST>> m_elements;
};

class IndexExprAST : public ExprAST {
public:
    IndexExprAST(SourceLocation loc, std::unique_ptr<ExprAST> target, std::unique_ptr<ExprAST> index)
        : ExprAST(loc), m_target(std::move(target)), m_index(std::move(index)) {}
    const ExprAST& target() const noexcept { return *m_target; }
    const ExprAST& index() const noexcept { return *m_index; }
    void accept(ASTVisitor& visitor) const override;
private:
    std::unique_ptr<ExprAST> m_target, m_index;
};
```

**Verification Tasks:**

1.  **Empty and Populated Literals:** Parse `[]` and
    `[1, 2 + 3, "four"]`. Verify that `ListLiteralExprAST` stores 0 and
    3 element subtrees respectively.
2.  **Chained Subscripting:** Parse `grid[x][y + 1]`. Verify that the
    root is an `IndexExprAST` whose target is another
    `IndexExprAST(grid, x)` and whose index is `BinaryExprAST(y + 1)`.
3.  **Unclosed Bracket Recovery:** Parse `arr[i + 1;`. Verify that the
    parser flags the missing `]` delimiter at the semicolon and triggers
    panic-mode synchronisation without crashing.

#### Exercise 19.3: Pattern Matching and Destructuring AST Grammar

**Context and Theory:**

Functional languages replace complex nested `if-else` cascades with
**Pattern Matching**. In Faberge, a `match` expression inspects a target
expression and dispatches to the first arm whose structural pattern
satisfies the value:

$$\text{MatchExpr} \to \text{"match"} \; \text{Expr} \; \text{"with"} \; \text{"\{"} \; \text{MatchArm}^* \; \text{"\}"}$$

$$\text{MatchArm} \to \text{Pattern} \; \text{"=>"} \; \text{Expr} \; \text{";"}^?$$

$$\text{Pattern} \to \text{Literal} \mid \text{Identifier} \mid \text{"\_"} \mid \text{"("} \; \text{Pattern} \; (\text{","} \; \text{Pattern})^* \; \text{")"}$$

A pattern can bind new variables in the arm's local scope (e.g. `(x, y)`
extracts tuple components), match exact scalar literals (e.g.
`0 => "zero"`), or match anything via a wildcard `_`.

```
match point with {
    (0, 0) => "origin",
    (x, 0) => "x-axis",
    _      => "generic point"
}
```

**Specification:**

Define a `PatternAST` base class with concrete subclasses:
`LiteralPatternAST`, `VariablePatternAST`, `WildcardPatternAST`, and
`TuplePatternAST`. Implement a `MatchExprAST` node storing the target
expression and a list of `MatchArm` descriptors. Extend the parser with
`parse_match_expression()` and `parse_pattern()`.

``` cpp
struct MatchArm {
    std::unique_ptr<PatternAST> pattern;
    std::unique_ptr<ExprAST> body;
};

class MatchExprAST : public ExprAST {
public:
    MatchExprAST(SourceLocation loc, std::unique_ptr<ExprAST> target, std::vector<MatchArm> arms)
        : ExprAST(loc), m_target(std::move(target)), m_arms(std::move(arms)) {}
    const ExprAST& target() const noexcept { return *m_target; }
    const std::vector<MatchArm>& arms() const noexcept { return m_arms; }
    void accept(ASTVisitor& visitor) const override;
private:
    std::unique_ptr<ExprAST> m_target;
    std::vector<MatchArm> m_arms;
};
```

**Verification Tasks:**

1.  **Wildcard and Literal Arm Parsing:** Parse
    `match x with { 1 => "one", _ => "other" }`. Verify that the
    resulting `MatchExprAST` contains exactly two arms, with the second
    arm possessing a `WildcardPatternAST`.
2.  **Tuple Destructuring Pattern:** Parse
    `match pair with { (a, b) => a + b }`. Verify that the arm's pattern
    parses as a `TuplePatternAST` containing two variable patterns (`a`
    and `b`).
3.  **Missing Fat Arrow Diagnostic:** Parse
    `match x with { 1 "error" }`. Verify that the parser detects the
    missing `=>` separator and reports a syntax error indicating the
    expected delimiter.

#### Exercise 19.4: Product Types: Heterogeneous Tuples

**Context and Theory:**

A **Tuple** represents an anonymous product type
$(v_1, v_2, \dots, v_n)$. In Faberge, tuples are declared using
comma-separated parenthesised expressions:

$$\text{TupleExpr} \to \text{"("} \; \text{Expr} \; \text{","} \; (\text{Expr} \; (\text{","} \; \text{Expr})^*)^? \; \text{")"} \mid \text{"("} \; \text{")"}$$

This construct introduces a classic parsing ambiguity with grouping
parentheses:

- `(x + 1)` is a grouped arithmetic expression containing a single inner
  AST node.
- `(x + 1,)` or `(x, y)` is a 2-tuple constructing a product value.
- `()` is a 0-tuple (the `Unit` or `Void` value).

The parser must disambiguate this at the opening parenthesis: parse the
first expression, and inspect the subsequent token. If followed by a
comma `,`, it transitions to tuple parsing; if followed by `)`, it
unwraps and returns the single grouped expression (see Figure 19–5).

<figure id="fig:parse-tuple">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2NjMiIGhlaWdo
dD0iOTcuNjAwMDA2IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDY2MyA5Ny42MDAwMDYi
PgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5z
dmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13
aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJv
a2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0
ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBD
b3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNw
eDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJh
bnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoK
LnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsK
ICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmls
bCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewog
IG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlh
bW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFy
a2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAu
ZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0K
Ci5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Bl
bl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtl
ci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29w
ZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNi
aWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJy
b3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9Ijci
IG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9s
eWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAg
ICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9
IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1y
ZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3Bv
bHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAw
IDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIg
b3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIg
cj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIg
aWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJr
ZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+
CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2Np
cmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3
Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVp
Z2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQi
IGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgog
IDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNjYz
IiBoZWlnaHQ9Ijk3LjYwMDAwNiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjkuNzUiIHk9IjExLjcwMDAwMSIgPlRva2VuPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjU2LjU1MDAwMyIgeT0iMTEuNzAwMDAxIiA+U3RyZWFtOjwvdGV4dD4KICA8
bGluZSB4MT0iMTYzLjgiIHkxPSI3LjgiIHgyPSIxODcuMjAwMDEiIHkyPSI3LjgiIGNsYXNzPSJz
b2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMTg3LjIwMDAxLDMuOSAxOTUsNy44IDE4
Ny4yMDAwMSwxMS43MDAwMDEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIyMDQuNzUiIHk9IjExLjcwMDAwMSIgPkV4cHI8L3RleHQ+CiAg
PGxpbmUgeDE9IjI0MS44IiB5MT0iNy44IiB4Mj0iMjY1LjIiIHkyPSI3LjgiIGNsYXNzPSJzb2xp
ZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMjY1LjIsMy45IDI3Myw3LjggMjY1LjIsMTEu
NzAwMDAxIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjMxOS44MDAwMiIg
eTE9IjcuOCIgeDI9IjM0My4yIiB5Mj0iNy44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9s
eWdvbiBwb2ludHM9IjM0My4yLDMuOSAzNTEsNy44IDM0My4yLDExLjcwMDAwMSIgY2xhc3M9ImZp
bGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM2MC43NSIg
eT0iMTEuNzAwMDAxIiA+VHVwbGVFeHByQVNUPC90ZXh0PgogIDxsaW5lIHgxPSIxNjMuOCIgeTE9
IjM5IiB4Mj0iMTg3LjIwMDAxIiB5Mj0iMzkiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5
Z29uIHBvaW50cz0iMTg3LjIwMDAxLDM1LjEwMDAwMiAxOTUsMzkgMTg3LjIwMDAxLDQyLjkiIGNs
YXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIy
MDQuNzUiIHk9IjQyLjkiID5FeHByPC90ZXh0PgogIDxsaW5lIHgxPSIyNDEuOCIgeTE9IjM5IiB4
Mj0iMjY1LjIiIHkyPSIzOSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRz
PSIyNjUuMiwzNS4xMDAwMDIgMjczLDM5IDI2NS4yLDQyLjkiIGNsYXNzPSJmaWxsZWQiPjwvcG9s
eWdvbj4KICA8bGluZSB4MT0iMzE5LjgwMDAyIiB5MT0iMzkiIHgyPSIzNDMuMiIgeTI9IjM5IiBj
bGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjM0My4yLDM1LjEwMDAwMiAz
NTEsMzkgMzQzLjIsNDIuOSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjM2MC43NSIgeT0iNDIuOSIgPkdyb3VwZWQ8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDIzLjE1MDAyIiB5PSI0Mi45IiA+RXhwcmVzc2lv
bjwvdGV4dD4KICA8bGluZSB4MT0iMTYzLjgiIHkxPSI3MC4yMDAwMDUiIHgyPSIyNjUuMiIgeTI9
IjcwLjIwMDAwNSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyNjUu
Miw2Ni4zIDI3Myw3MC4yMDAwMDUgMjY1LjIsNzQuMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29u
PgogIDxsaW5lIHgxPSIzMTkuODAwMDIiIHkxPSI3MC4yMDAwMDUiIHgyPSIzNDMuMiIgeTI9Ijcw
LjIwMDAwNSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIzNDMuMiw2
Ni4zIDM1MSw3MC4yMDAwMDUgMzQzLjIsNzQuMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM2MC43NSIgeT0iNzQuMSIgPlR1cGxlRXhw
ckFTVDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMjYuNzUiIHk9IjEx
LjcwMDAwMSIgPiYjMzk7KCYjMzk7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjI4Mi43NSIgeT0iMTEuNzAwMDAxIiA+JiMzOTssJiMzOTs8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDYyLjE1MDAyIiB5PSIxMS43MDAwMDEiID4oTXVsdGktZWxl
bWVudCBwcm9kdWN0KTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMjYu
NzUiIHk9IjQyLjkiID4mIzM5OygmIzM5OzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIyODIuNzUiIHk9IjQyLjkiID4mIzM5OykmIzM5OzwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSI1MDguOTUiIHk9IjQyLjkiID4oVW53cmFwcGVkIEV4cHJBU1Qp
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEyNi43NSIgeT0iNzQuMSIg
PiYjMzk7KCYjMzk7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI4Mi43
NSIgeT0iNzQuMSIgPiYjMzk7KSYjMzk7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjQ2Mi4xNTAwMiIgeT0iNzQuMSIgPigwLXR1cGxlIC8gVW5pdCk8L3RleHQ+Cjwvc3Zn
Pgo=
" alt="" />
<figcaption>Figure 19–5: Correct Parsing of Tuples</figcaption>
</figure>

**Specification:**

Create a `TupleExprAST` class storing a
`std::vector<std::unique_ptr<ExprAST>>`. Refactor `parse_prefix()` when
consuming `TokenType::LeftParen` to distinguish cleanly between unit
literals, parenthesised expressions, and multi-element tuples.

``` cpp
class TupleExprAST : public ExprAST {
public:
    TupleExprAST(SourceLocation loc, std::vector<std::unique_ptr<ExprAST>> elements)
        : ExprAST(loc), m_elements(std::move(elements)) {}
    const auto& elements() const noexcept { return m_elements; }
    [[nodiscard]] size_t arity() const noexcept { return m_elements.size(); }
    void accept(ASTVisitor& visitor) const override;
private:
    std::vector<std::unique_ptr<ExprAST>> m_elements;
};
```

**Verification Tasks:**

1.  **Unit Literal Disambiguation:** Parse `()`. Verify that the parser
    emits a `TupleExprAST` with `arity() == 0`.
2.  **Grouping vs Tuple Invariant:** Parse `(42)` and `(42,)`. Verify
    that the former yields a raw `LiteralExprAST(42)` without tuple
    wrapping, while the latter yields a `TupleExprAST` of arity 1.
3.  **Nested Heterogeneous Products:** Parse `(1, "two", (3.0, true))`.
    Verify that the root tuple has arity 3, and its third element is a
    nested `TupleExprAST` of arity 2.

#### Exercise 19.5: Postfix Factorial and Unary Increment Operators

**Context and Theory:**

Standard unary operators (such as negative `-x` or logical negation
`!x`) are **prefix operators**, preceding their operand. Conversely,
operators such as postfix factorial ($n!$) or increment ($x\text{++}$)
are **postfix operators**, trailing their operand.

In Pratt parsing, postfix operators are handled as a special case of the
infix denotation (`led`, see Figure 19–6):

1.  A postfix operator appears after an expression: it is registered in
    the infix operator table with very high precedence ($85$).
2.  Unlike a binary infix operator (which consumes a right-hand
    operand), a postfix operator **consumes no further expressions**. It
    wraps the existing `left` AST node into a `UnaryExprAST` (or
    `PostfixExprAST`) and returns immediately.

<figure id="fig:parse-postfix">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NzcuMiIgaGVp
Z2h0PSIxNjAiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNTc3LjIgMTYwIj4KICA8c3R5
bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xlLCAuc3ZnYm9iIHJl
Y3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJva2Utd2lkdGg6IDEu
NTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAgc3Ryb2tlLWxpbmVj
YXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdib2IgdGV4dCB7CiAg
d2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWlseTogQ291cmllciBO
ZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6IDEzcHg7Cn0KCi5z
dmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6IHRyYW5zcGFyZW50
Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7Cn0KCi5zdmdib2Ig
LmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxlZCB7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5ub2ZpbGwgewogIGZp
bGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93IHsKICBtYXJrZXIt
ZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fycm93IHsKICBtYXJr
ZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2RpYW1vbmQgewog
IG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0
X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI29wZW5fY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6
IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2JpZ19vcGVuX2NpcmNs
ZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYmlnX29wZW5f
Y2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9ImFycm93IiB2aWV3
Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJI
ZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9p
bnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtl
ciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJr
ZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+
CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+PC9wb2x5Z29uPgog
ICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJl
Zlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0i
YXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNs
YXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJvcGVu
X2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAg
ICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIgdmlld0JveD0iMCAw
IDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIg
b3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIg
cj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICA8L2RlZnM+
CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9IjU3Ny4yIiBoZWln
aHQ9IjE2MCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9
IjExLjcwMDAwMSIgPlN0cmVhbTo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iNzkuOTUwMDA1IiB5PSIxMS43MDAwMDEiID5uPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9Ijk1LjU1IiB5PSIxMS43MDAwMDEiID4hPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjExMS4xNSIgeT0iMTEuNzAwMDAxIiA+KzwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMjYuNzUiIHk9IjExLjcwMDAwMSIgPjE8L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTcuNTUwMDAxIiB5PSIxMjAuOSIgPlN0
ZXA8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTYuNTUwMDAzIiB5PSIx
MjAuOSIgPjM6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijc5Ljk1MDAw
NSIgeT0iMTIwLjkiID4mIzM5OzwvdGV4dD4KICA8bGluZSB4MT0iODUuOCIgeTE9IjExNyIgeDI9
IjkzLjYwMDAwNiIgeTI9IjExNyIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iOTUuNTUiIHk9IjEyMC45IiA+JiMzOTs8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTExLjE1IiB5PSIxMjAuOSIgPmVuY291bnRlcmVkLjwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMTIuNTUiIHk9IjEyMC45IiA+
UHJlY2VkZW5jZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOTguMzUi
IHk9IjEyMC45IiA+NTA8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzIx
Ljc1IiB5PSIxMjAuOSIgPiZsdDs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMzM3LjM1IiB5PSIxMjAuOSIgPjg1LjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI3OS45NTAwMDUiIHk9IjEzNi41IiA+V3JhcHM8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMTI2Ljc1IiB5PSIxMzYuNSIgPlBvc3RmaXhFeHByQVNUPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI0My43NSIgeT0iMTM2LjUiID5hbmQ8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjc0Ljk1IiB5PSIxMzYuNSIg
PnBhcnNlczwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjkuNTUwMDIi
IHk9IjEzNi41IiA+UkhTPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM2
MC43NSIgeT0iMTM2LjUiID4mIzM5OzEmIzM5Oy48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMTcuNTUwMDAxIiB5PSI0Mi45IiA+U3RlcCAxOiAmIzM5O24mIzM5OyBwYXJz
ZWQgYXMgUHJpbWFyeSAobnVkKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIxNy41NTAwMDEiIHk9Ijc0LjEiID5TdGVwIDI6ICYjMzk7ISYjMzk7IGVuY291bnRlcmVkLiBQ
cmVjZWRlbmNlIDg1ICZndDsgMC48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iNzkuOTUwMDA1IiB5PSI4OS43MDAwMDUiID5XcmFwcyAmIzM5O24mIzM5OyBpbnRvIFBvc3Rm
aXhFeHByQVNUKG4sICEpLiBDb25zdW1lcyBubyByaWdodCBvcGVyYW5kITwvdGV4dD4KPC9zdmc+
Cg==
" alt="" />
<figcaption>Figure 19–6: Precedence-Based Parsing of Postfix
Operator</figcaption>
</figure>

**Specification:**

Implement a `PostfixExprAST` node. Register postfix operators
`TokenType::Exclamation` (when trailing an expression) and
`TokenType::PlusPlus` in the Pratt parser's infix table. Implement the
zero-RHS consumption logic inside `parse_infix()`.

``` cpp
class PostfixExprAST : public ExprAST {
public:
    PostfixExprAST(SourceLocation loc, TokenType op, std::unique_ptr<ExprAST> operand)
        : ExprAST(loc), m_op(op), m_operand(std::move(operand)) {}
    TokenType op() const noexcept { return m_op; }
    const ExprAST& operand() const noexcept { return *m_operand; }
    void accept(ASTVisitor& visitor) const override;
private:
    TokenType m_op;
    std::unique_ptr<ExprAST> m_operand;
};
```

**Verification Tasks:**

1.  **Postfix vs Prefix Invariance:** Parse `!x!`. Verify that the root
    is a prefix `UnaryExprAST(!)` whose operand is a
    `PostfixExprAST(x, !)`.
2.  **Precedence over Binary Operators:** Parse `n! * 2`. Verify that
    the multiplication operator binds above the factorial, yielding
    `BinaryExprAST(PostfixExprAST(n, !), *, 2)`.
3.  **Chained Postfix Execution:** Parse `n!!`. Verify that the AST
    groups into `PostfixExprAST(PostfixExprAST(n, !), !)`.

#### Exercise 19.6: High-Fidelity JSON AST Serialiser (Visitor Engine)

**Context and Theory:**

Modern compiler toolchains require structural interchange formats for
language servers (LSP), static analysers, and downstream code emitters
(such as the Faberge C++ emitter).

The **Visitor Pattern** enables recursive AST traversal without mutating
node classes. A `JsonASTSerializer` visits every node in the AST and
generates a well-formed JSON object:

- `kind`: The concrete node type name (e.g. `"LetStmt"`,
  `"BinaryExpr"`).
- `location`: The starting line, column, and character offset.
- Node-specific attributes (e.g. `operator`, `variable_name`,
  `literal_value`).
- Nested child subtrees recursively serialised into JSON arrays or
  sub-objects.

To demonstrate, the JSON Serialisation of `LetStmtAST('x', Literal(10))`
should be:

``` json
 {
   "kind": "LetStmt",
   "name": "x",
   "loc": { "line": 1, "col": 1 },
   "init": { "kind": "Literal", "type": "Int", "value": "10" }
 }
```

**Specification:**

Implement a `JsonASTSerializer` class inheriting from `ASTVisitor`.
Provide an entry method `serialize(const ASTNode&)` returning a
formatted JSON string representing the entire AST hierarchy.

``` cpp
class JsonASTSerializer : public ASTVisitor {
public:
    std::string serialize(const ASTNode& node) {
        m_buffer.str("");
        m_buffer.clear();
        node.accept(*this);
        return m_buffer.str();
    }
    void visit(const LetStmtAST& node) override;
    void visit(const BinaryExprAST& node) override;
    void visit(const LiteralExprAST& node) override;
    // ... Implement remaining node visitors ...
private:
    std::ostringstream m_buffer;
    void emit_location(SourceLocation loc);
};
```

**Verification Tasks:**

1.  **Valid JSON Formatting:** Serialise a program containing variables,
    lambdas, and conditionals. Parse the resulting string with a
    standard JSON validator and assert zero parsing errors.
2.  **Location Accuracy:** Verify that the JSON output contains `"line"`
    and `"column"` keys matching the source coordinate of every
    expression.
3.  **Full Subtree Traversal:** Serialise a complex arithmetic
    expression tree `1 + 2 * 3`. Verify that the JSON reflects the
    binary tree structure with `"left"` and `"right"` child keys.

#### Exercise 19.7: Ternary Conditional Operator Integration (`? :`)

**Context and Theory:**

Many programming languages provide the ternary conditional operator as a
concise alternative to `if-then-else`:

$$\text{TernaryExpr} \to \text{Condition} \; \text{"?"} \; \text{ThenExpr} \; \text{":"} \; \text{ElseExpr}$$

The ternary operator is a **mixfix (ternary) operator** requiring two
separate delimiters (`?` and `:`). In Pratt parsing (as shown in
Figure 19–7):

1.  `?` is treated as an infix operator with low precedence ($25$,
    binding just above assignment).
2.  When `?` is encountered, the parser consumes the `then_branch`
    expression with precedence $0$ (allowing any nested expression).
3.  The parser strictly asserts and consumes the `:` delimiter.
4.  The `else_branch` expression is parsed with precedence $24$ (one
    less than the ternary precedence) to ensure **right-associativity**:

$$c_1 \mathbin{?} e_1 \mathbin{:} c_2 \mathbin{?} e_2 \mathbin{:} e_3 \equiv c_1 \mathbin{?} e_1 \mathbin{:} (c_2 \mathbin{?} e_2 \mathbin{:} e_3)$$

<figure id="fig:parse-ternary">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0NjAuMiIgaGVp
Z2h0PSI4MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA0NjAuMiA4MiI+CiAgPHN0eWxl
Pi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwgLnN2Z2JvYiByZWN0
LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tlLXdpZHRoOiAxLjU7
CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0cm9rZS1saW5lY2Fw
OiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9iIHRleHQgewogIHdo
aXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6IENvdXJpZXIgTmV3
LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxM3B4Owp9Cgouc3Zn
Ym9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFuc3BhcmVudDsK
fQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgouc3ZnYm9iIC5m
aWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewogIGZpbGw6IHRy
YW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxsIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAgbWFya2VyLWVu
ZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFtb25kIHsKICBt
YXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfZGlhbW9u
ZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2Vk
X2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
ZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVuX2NpcmNsZSk7
Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1
cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Blbl9jaXJjbGUg
ewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2JpZ19vcGVuX2Np
cmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJvdyIgdmlld0Jv
eD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVp
Z2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29uIHBvaW50
cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIg
aWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2Vy
V2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgog
ICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9seWdvbj4KICAg
IDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZY
PSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1
dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFz
cz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0ib3Blbl9j
aXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3
IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNp
cmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAg
PC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4
IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9y
aWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9
IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgPC9kZWZzPgog
IDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI0NjAuMiIgaGVpZ2h0
PSI4MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9IjEx
LjcwMDAwMSIgPlN0cmVhbTo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
NzIuMTUiIHk9IjExLjcwMDAwMSIgPmNvbmQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMTExLjE1IiB5PSIxMS43MDAwMDEiID4/PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjEyNi43NSIgeT0iMTEuNzAwMDAxIiA+QTwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNTcuOTUiIHk9IjExLjcwMDAwMSIgPkI8L3RleHQ+CiAg
PHBvbHlnb24gcG9pbnRzPSIxODcuMjAwMDEsMzUuMTAwMDAyIDE5NSwzOSAxODcuMjAwMDEsNDIu
OSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxwb2x5Z29uIHBvaW50cz0iMTg3LjIwMDAx
LDUwLjcgMTk1LDU0LjYwMDAwMiAxODcuMjAwMDEsNTguNSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5
Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE0Mi4zNSIgeT0iMTEuNzAwMDAx
IiA+OjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMDQuNzUiIHk9IjQy
LjkiID5SSFM8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjA0Ljc1IiB5
PSI1OC41IiA+TWlkPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjcyLjE1
IiB5PSI1OC41IiA+TEhTPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIz
NS45NTAwMSIgeT0iNDIuOSIgPihQYXJzZWQgYWZ0ZXIgJiMzOTs6JiMzOTspPC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIzNS45NTAwMSIgeT0iNTguNSIgPihQYXJzZWQg
YmV0d2VlbiAmIzM5Oz8mIzM5OyBhbmQgJiMzOTs6JiMzOTspPC90ZXh0PgogIDxnPgogICAgPGxp
bmUgeDE9IjcwLjIwMDAwNSIgeTE9IjE1LjYiIHgyPSI3OCIgeTI9IjMxLjIiIGNsYXNzPSJzb2xp
ZCI+PC9saW5lPgogICAgPGxpbmUgeDE9Ijc4IiB5MT0iMzEuMiIgeDI9IjkzLjYwMDAwNiIgeTI9
IjMxLjIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjEwMS40IiB5MT0iMTUu
NiIgeDI9IjkzLjYwMDAwNiIgeTI9IjMxLjIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwvZz4K
ICA8Zz4KICAgIDxsaW5lIHgxPSIxMjguNyIgeTE9IjE1LjYiIHgyPSIxMjguNyIgeTI9IjU0LjYw
MDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMTI4LjciIHkxPSI1NC42
MDAwMDIiIHgyPSIxODcuMjAwMDEiIHkyPSI1NC42MDAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5l
PgogIDwvZz4KICA8Zz4KICAgIDxsaW5lIHgxPSIxNTkuOTAwMDEiIHkxPSIxNS42IiB4Mj0iMTU5
LjkwMDAxIiB5Mj0iMzkiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjE1OS45
MDAwMSIgeTE9IjM5IiB4Mj0iMTg3LjIwMDAxIiB5Mj0iMzkiIGNsYXNzPSJzb2xpZCI+PC9saW5l
PgogIDwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 19–7: Correct Parsing of Ternary
Operator</figcaption>
</figure>

**Specification:**

Implement a `TernaryExprAST` node class. Extend Faberge's Pratt parser
to register `TokenType::Question` in the infix table. Parse the middle
expression, consume `TokenType::Colon`, and recursively parse the right
branch with right-associative binding power.

``` cpp
class TernaryExprAST : public ExprAST {
public:
    TernaryExprAST(SourceLocation loc, std::unique_ptr<ExprAST> cond,
                   std::unique_ptr<ExprAST> then_expr, std::unique_ptr<ExprAST> else_expr)
        : ExprAST(loc), m_condition(std::move(cond)), 
          m_then_expr(std::move(then_expr)), m_else_expr(std::move(else_expr)) {}
    const ExprAST& condition() const noexcept { return *m_condition; }
    const ExprAST& then_expr() const noexcept { return *m_then_expr; }
    const ExprAST& else_expr() const noexcept { return *m_else_expr; }
    void accept(ASTVisitor& visitor) const override;
private:
    std::unique_ptr<ExprAST> m_condition, m_then_expr, m_else_expr;
};
```

**Verification Tasks:**

1.  **Standard Ternary Parsing:** Parse `x > 0 ? 1 : -1`. Verify that
    the resulting `TernaryExprAST` accurately separates condition,
    then-expression, and else-expression.
2.  **Right-Associativity Chaining:** Parse `a ? 1 : b ? 2 : 3`. Verify
    that the root else-branch contains another nested
    `TernaryExprAST(b ? 2 : 3)`.
3.  **Missing Colon Syntax Recovery:** Parse `a ? 1 ;`. Verify that the
    parser flags the missing `:` delimiter at the semicolon, records a
    syntax diagnostic, and recovers gracefully.

#### Exercise 19.8: AST Linting: Redundant Grouping Elimination

**Context and Theory:**

Developers frequently introduce redundant grouping parentheses, either
out of habit or uncertainty over operator precedence:

```
let result = ((x * y)) + (z);
```

While harmless to evaluation semantics, redundant groupings clutter AST
structures, inflate memory usage, and introduce noise into downstream
code emitters.

An **AST Linter Pass** traverses the tree and identifies redundant
parentheses:

1.  **Duplicate Enclosure:** Any expression directly wrapped in multiple
    consecutive grouping parentheses `((e))`.
2.  **Natural Precedence Enclosure:** A sub-expression enclosed in
    parentheses whose operator naturally binds tighter than its parent
    operator (e.g. `(a * b) + c`, where `*` already has higher
    precedence than `+`).
3.  **Essential Grouping:** Parentheses that actively override
    precedence (e.g. `(a + b) * c`) are non-redundant and must be
    preserved.

- **Redundant:** `(a * b) + c` → Naturally binds as: `a * b + c`
- **Essential:** `(a + b) * c` → Overrides natural precedence! Must NOT
  remove!

**Specification:**

Create an `ASTLintVisitor` class that traverses expressions, tracks
parent and child operator precedences, and reports diagnostics with
`FixItHint` replacements pointing out redundant parentheses.

``` cpp
class ASTLintVisitor : public ASTVisitor {
public:
    void check(const ASTNode& root);
    size_t redundant_parentheses_count() const noexcept { return m_redundant_count; }
    void visit(const BinaryExprAST& node) override;
    // ... Implement visitor methods ...
private:
    size_t m_redundant_count{0};
    void inspect_child_precedence(TokenType parent_op, const ExprAST& child, bool is_right_child);
};
```

**Verification Tasks:**

1.  **Duplicate Parenthesis Detection:** Lint `let x = (((42)));`.
    Verify that the linter reports exactly two redundant grouping
    warnings with fix-it deletions.
2.  **Natural Precedence Warning:** Lint `(a * b) + c`. Verify that the
    linter flags the parentheses around `a * b` as redundant.
3.  **Essential Parenthesis Preservation:** Lint `(a + b) * c`. Verify
    that the linter identifies the grouping as essential and emits zero
    warnings.

#### Exercise 19.9: High-Performance Arena Allocator for AST Nodes

**Context and Theory:**

In a standard compiler architecture, every AST node is individually
allocated on the free store via `std::make_unique<Node>()`. For a
$100,000$-line codebase containing millions of AST nodes, this incurs
massive performance penalties:

- High heap allocation overhead (`malloc`/`free` book-keeping).
- Pointer fragmentation across physical RAM, resulting in CPU cache
  misses during tree traversals.

Production compilers (such as Clang and rustc) employ **Arena (Bump)
Allocators** (see Figure 19–8):

1.  Memory is pre-allocated in large contiguous memory regions (e.g.
    $4\text{ MB}$ chunks).
2.  Creating a node involves advancing a bump pointer by `sizeof(Node)`
    aligned to 8-byte boundaries ($O(1)$ allocation).
3.  Individual nodes are never destructed one-by-one. When the
    compilation pass completes, the entire arena memory pool is
    deallocated or reset in a single operation ($O(1)$ teardown).

<figure id="fig:arena-ast">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NjkuNCIgaGVp
Z2h0PSI2Ni40IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDU2OS40IDY2LjQiPgogIDxz
dHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5zdmdib2Ig
cmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13aWR0aDog
MS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJva2UtbGlu
ZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0ZXh0IHsK
ICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNweDsKfQoK
LnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNwYXJl
bnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2Z2Jv
YiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1hcmtl
ci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9uZCB7
CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5kX21h
cmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9jaXJj
bGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5fY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0
YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdfb3Bl
bl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ciIHZp
ZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBw
b2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHlnb24+
CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIg
Y2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9Im9w
ZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4K
ICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwvZGVm
cz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNTY5LjQiIGhl
aWdodD0iNjYuNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUi
IHk9IjExLjcwMDAwMSIgPkFyZW5hPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjU2LjU1MDAwMyIgeT0iMTEuNzAwMDAxIiA+TWVtb3J5PC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjExMS4xNSIgeT0iMTEuNzAwMDAxIiA+QmxvY2s8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTU3Ljk1IiB5PSIxMS43MDAwMDEiID5bNDwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxODEuMzUiIHk9IjExLjcwMDAw
MSIgPk1CXTo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOS43NSIgeT0i
NDIuOSIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjUuMzUiIHk9
IjQyLjkiID5Ob2RlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjY0LjM1
IiB5PSI0Mi45IiA+MTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3OS45
NTAwMDUiIHk9IjQyLjkiID5dWzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIxMDMuMzUwMDA2IiB5PSI0Mi45IiA+Tm9kZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSIxNDIuMzUiIHk9IjQyLjkiID4yPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjE1Ny45NSIgeT0iNDIuOSIgPl1bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjE4MS4zNSIgeT0iNDIuOSIgPk5vZGU8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjIwLjM1IiB5PSI0Mi45IiA+MzwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMzUuOTUwMDEiIHk9IjQyLjkiID5dPC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1MS41NSIgeT0iNDIuOSIgPi4uLjwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyODIuNzUiIHk9IjQyLjkiID5bPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI5OC4zNSIgeT0iNDIuOSIgPkJ1bXA8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzM3LjM1IiB5PSI0Mi45IiA+
UG9pbnRlcjwvdGV4dD4KICA8bGluZSB4MT0iMzk3LjgwMDAyIiB5MT0iMzkiIHgyPSI0NTIuNDAw
MDIiIHkyPSIzOSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSI0NTIu
NDAwMDIsMzUuMTAwMDAyIDQ2MC4yLDM5IDQ1Mi40MDAwMiw0Mi45IiBjbGFzcz0iZmlsbGVkIj48
L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDY5Ljk1IiB5PSI0Mi45
IiA+RnJlZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1MDguOTUiIHk9
IjQyLjkiID5TcGFjZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1NTUu
NzUiIHk9IjQyLjkiID5dPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 19–8: AST Nodes Arena Allocator</figcaption>
</figure>

**Specification:**

Implement a custom `ASTArenaAllocator` supporting typed allocations via
placement `new`. Integrate it into the Faberge parser to allocate AST
nodes contiguously without individual heap calls.

``` cpp
class ASTArenaAllocator {
public:
    explicit ASTArenaAllocator(size_t chunk_size = 4096 * 1024);
    ~ASTArenaAllocator();

    template <typename T, typename... Args>
    T* allocate(Args&&... args) {
        size_t bytes = sizeof(T);
        size_t aligned_size = (bytes + 7) & ~7; // 8-byte alignment
        if (m_current_offset + aligned_size > m_chunk_size) allocate_new_chunk();
        void* ptr = m_current_chunk + m_current_offset;
        m_current_offset += aligned_size;
        return new (ptr) T(std::forward<Args>(args)...);
    }
    void reset();
private:
    size_t m_chunk_size;
    size_t m_current_offset{0};
    char* m_current_chunk{nullptr};
    std::vector<char*> m_allocated_chunks;
    void allocate_new_chunk();
};
```

**Verification Tasks:**

1.  **Contiguous Address Verification:** Allocate three consecutive
    `LiteralExprAST` nodes. Verify that the difference between their
    memory addresses matches their 8-byte aligned `sizeof` footprint.
2.  **Chunk Growth across Capacity:** Set chunk size to $1024$ bytes.
    Allocate $500$ nodes. Verify that the arena transparently allocates
    additional chunks without memory corruption or segmentation faults.
3.  **Instantaneous Teardown:** Measure the deallocation duration of
    $1,000,000$ AST nodes. Verify that resetting the arena executes in
    sub-millisecond time.

#### Exercise 19.10: Syntactic Error Recovery: Error Nodes and Synchronisation

**Context and Theory:**

A primitive compiler aborts upon encountering its first syntax error. A
production compiler must continue parsing after an error to discover
subsequent issues in a single pass, enabling IDE features like
autocompletion and diagnostics.

To recover from syntax errors without crashing or cascading false
errors, modern parsers use **Error Nodes** (see Figure 19–9):

1.  When parsing an expression fails, rather than throwing an exception
    that unwinds the entire call stack, the parser logs a diagnostic and
    constructs an `ErrorExprAST` node.
2.  The `ErrorExprAST` node is inserted directly into the AST,
    preserving the hierarchy of surrounding statements.
3.  The parser synchronises by advancing past unrecognised tokens until
    reaching an element in its **Synchronisation Set** (e.g. `;`, `}`,
    or keywords `let`, `fn`, `if`).

<figure id="fig:error-nodes">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1ODUiIGhlaWdo
dD0iMTI4LjgiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNTg1IDEyOC44Ij4KICA8c3R5
bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xlLCAuc3ZnYm9iIHJl
Y3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJva2Utd2lkdGg6IDEu
NTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAgc3Ryb2tlLWxpbmVj
YXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdib2IgdGV4dCB7CiAg
d2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWlseTogQ291cmllciBO
ZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6IDEzcHg7Cn0KCi5z
dmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6IHRyYW5zcGFyZW50
Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7Cn0KCi5zdmdib2Ig
LmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxlZCB7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5ub2ZpbGwgewogIGZp
bGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93IHsKICBtYXJrZXIt
ZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fycm93IHsKICBtYXJr
ZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2RpYW1vbmQgewog
IG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0
X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI29wZW5fY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6
IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2JpZ19vcGVuX2NpcmNs
ZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYmlnX29wZW5f
Y2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9ImFycm93IiB2aWV3
Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJI
ZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9p
bnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtl
ciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJr
ZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+
CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+PC9wb2x5Z29uPgog
ICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJl
Zlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0i
YXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNs
YXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJvcGVu
X2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAg
ICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIgdmlld0JveD0iMCAw
IDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIg
b3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIg
cj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICA8L2RlZnM+
CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9IjU4NSIgaGVpZ2h0
PSIxMjguOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUiIHk9
IjExLjcwMDAwMSIgPlNvdXJjZTo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iODcuNzUiIHk9IjExLjcwMDAwMSIgPmxldDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSIxMTguOTUwMDA1IiB5PSIxMS43MDAwMDEiID5hPC90ZXh0PgogIDxsaW5lIHgx
PSIxMzIuNiIgeTE9IjUuODUwMDAwNCIgeDI9IjE0MC40MDAwMSIgeTI9IjUuODUwMDAwNCIgY2xh
c3M9InNvbGlkIj48L2xpbmU+CiAgPGxpbmUgeDE9IjEzMi42IiB5MT0iOS43NSIgeDI9IjE0MC40
MDAwMSIgeTI9IjkuNzUiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjE1MC4xNTAwMSIgeT0iMTEuNzAwMDAxIiA+MTA7PC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE5Ni45NTAwMSIgeT0iMTEuNzAwMDAxIiA+bGV0PC90
ZXh0PgogIDxsaW5lIHgxPSIyMTQuNSIgeTE9IjI3LjMwMDAwMSIgeDI9IjIxNC41IiB5Mj0iNTAu
NyIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MTgxLjM1IiB5PSI3NC4xIiA+U3ludGF4PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjIyOC4xNTAwMSIgeT0iMTEuNzAwMDAxIiA+YjwvdGV4dD4KICA8bGluZSB4MT0iMjQx
LjgiIHkxPSI1Ljg1MDAwMDQiIHgyPSIyNDkuNiIgeTI9IjUuODUwMDAwNCIgY2xhc3M9InNvbGlk
Ij48L2xpbmU+CiAgPGxpbmUgeDE9IjI0MS44IiB5MT0iOS43NSIgeDI9IjI0OS42IiB5Mj0iOS43
NSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MjU5LjM1IiB5PSIxMS43MDAwMDEiID4qPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjI3NC45NSIgeT0iMTEuNzAwMDAxIiA+OzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIzMDYuMTUiIHk9IjExLjcwMDAwMSIgPmxldDwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMzcuMzUiIHk9IjExLjcwMDAwMSIgPmM8L3RleHQ+CiAg
PGxpbmUgeDE9IjM1MSIgeTE9IjUuODUwMDAwNCIgeDI9IjM1OC44MDAwMiIgeTI9IjUuODUwMDAw
NCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPGxpbmUgeDE9IjM1MSIgeTE9IjkuNzUiIHgyPSIz
NTguODAwMDIiIHkyPSI5Ljc1IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSIzNjguNTUwMDIiIHk9IjExLjcwMDAwMSIgPjIwOzwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Ljc1IiB5PSI3NC4xIiA+UGFyc2VyOjwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMzUuOTUwMDEiIHk9Ijc0LjEiID5F
cnJvciE8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOS43NSIgeT0iMTA1
LjMiID5BU1Q6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijg3Ljc1IiB5
PSIxMDUuMyIgPltMZXRTdG10OjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIxNjUuNzUiIHk9IjEwNS4zIiA+YV08L3RleHQ+CiAgPGxpbmUgeDE9IjE4Ny4yMDAwMSIgeTE9
IjEwMS40IiB4Mj0iMjAyLjgiIHkyPSIxMDEuNCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBv
bHlnb24gcG9pbnRzPSIyMDIuOCw5Ny41IDIxMC42LDEwMS40IDIwMi44LDEwNS4zIiBjbGFzcz0i
ZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjIwLjM1
IiB5PSIxMDUuMyIgPltMZXRTdG10OjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIyOTguMzUiIHk9IjEwNS4zIiA+Yiw8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMzIxLjc1IiB5PSIxMDUuMyIgPkluaXQ6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjM2OC41NTAwMiIgeT0iMTA1LjMiID5FcnJvck5vZGVdPC90ZXh0Pgog
IDxsaW5lIHgxPSI0NTIuNDAwMDIiIHkxPSIxMDEuNCIgeDI9IjQ2OCIgeTI9IjEwMS40IiBjbGFz
cz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjQ2OCw5Ny41IDQ3NS44MDAwMiwx
MDEuNCA0NjgsMTA1LjMiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSI0ODUuNTUwMDIiIHk9IjEwNS4zIiA+W0xldFN0bXQ6PC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjU2My41NSIgeT0iMTA1LjMiID5jXTwvdGV4
dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 19–9: Syntactic Error Recovery with Error
Nodes</figcaption>
</figure>

**Specification:**

Define an `ErrorExprAST` class. Refactor `Parser::parse_expression()`
and statement parsing routines: upon encountering a malformed
expression, emit a diagnostic, construct an `ErrorExprAST`, synchronise
to the next statement delimiter, and resume normal parsing.

``` cpp
class ErrorExprAST : public ExprAST {
public:
    explicit ErrorExprAST(SourceLocation loc) : ExprAST(loc) {}
    void accept(ASTVisitor& visitor) const override;
};

class ResilientParser {
public:
    std::unique_ptr<ExprAST> parse_expression_resilient(int min_prec) {
        try {
            return parse_expression(min_prec);
        } catch (const std::exception& e) {
            SourceLocation err_loc = current_token().location;
            synchronise_to_expression_boundary();
            return std::make_unique<ErrorExprAST>(err_loc);
        }
    }
private:
    void synchronise_to_expression_boundary();
};
```

**Verification Tasks:**

1.  **Multi-Error Discovery:** Parse a file containing three separate
    syntax errors across three distinct function bodies. Verify that the
    parser reports all three errors and successfully parses the final
    valid statement.
2.  **Error Node AST Retention:** Inspect the generated AST of a file
    containing a malformed variable initialiser. Verify that the
    statement list contains an entry with an `ErrorExprAST` child node.
3.  **Infinite Loop Immunity:** Feed a stream consisting of entirely
    invalid punctuation `!@#$%^&*()`. Verify that the synchronisation
    loop consumes tokens toward `EOF` without locking into an infinite
    loop.

#### Exercise 19.11: Precedence Climbing vs Pratt Parsing

**Context and Theory:**

**Precedence Climbing** is an algorithmically compact alternative to
Pratt parsing. While Pratt parsing models operator rules through
separate prefix (`nud`) and infix (`led`) denotation dispatch tables,
Precedence Climbing formulates binary expression parsing as an iterative
loop driven by operator binding queries (see Figure 19–10):

$$\text{Climb}(\text{min\_prec}):$$

1.  Parse primary/prefix operand $L$.
2.  While current token is binary operator with
    $\text{prec} \ge \text{min\_prec}$:
    - Fetch precedence $p$ and associativity of operator.
    - Compute next precedence threshold:
      $p' = (\text{is\_right\_assoc}) \mathbin{?} p \mathbin{:} p + 1$.
    - Recursively parse right-hand operand:
      $R \leftarrow \text{Climb}(p')$.
    - Update $L \leftarrow \text{BinaryExpr}(L, \text{op}, R)$.
3.  Return $L$.

Both algorithms produce isomorphic Abstract Syntax Trees.

<figure id="fig:precedence-climbing">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI3MDVwdCIgaGVpZ2h0PSI0NHB0Igogdmlld0JveD0iMC4wMCAwLjAw
IDcwNS4wNSA0NC4wMCIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIiB4bWxuczp4
bGluaz0iaHR0cDovL3d3dy53My5vcmcvMTk5OS94bGluayI+CjxnIGlkPSJncmFwaDAiIGNsYXNz
PSJncmFwaCIgdHJhbnNmb3JtPSJzY2FsZSgxIDEpIHJvdGF0ZSgwKSB0cmFuc2xhdGUoNCA0MCki
Pgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gcHJpbWFyeSAtLT4KPGcgaWQ9Im5vZGUxIiBjbGFzcz0i
bm9kZSI+Cjx0aXRsZT5wcmltYXJ5PC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tl
PSJibGFjayIgY3g9IjU1LjkyIiBjeT0iLTE4IiByeD0iNTUuODUiIHJ5PSIxOCIvPgo8dGV4dCB0
ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI1NS45MiIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJDb3Vy
aWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5b
IFByaW1hcnkgTCBdPC90ZXh0Pgo8L2c+CjwhLS0gd2hpbGUgLS0+CjxnIGlkPSJub2RlMiIgY2xh
c3M9Im5vZGUiPgo8dGl0bGU+d2hpbGU8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJv
a2U9ImJsYWNrIiBjeD0iMjM5LjIyIiBjeT0iLTE4IiByeD0iMTA1LjI0IiByeT0iMTgiLz4KPHRl
eHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjM5LjIyIiB5PSItMTUuNSIgZm9udC1mYW1pbHk9
IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAu
MDAiPldoaWxlIChvcC5wcmVjICZndDs9IG1pbl9wcmVjKTwvdGV4dD4KPC9nPgo8IS0tIHByaW1h
cnkmIzQ1OyZndDt3aGlsZSAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5w
cmltYXJ5JiM0NTsmZ3Q7d2hpbGU8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBkPSJNMTEyLjMsLTE4QzExNS45NSwtMTggMTE5LjY5LC0xOCAxMjMuNSwtMTgiLz4KPHBv
bHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTIzLjY4LC0yMS41IDEz
My42OCwtMTggMTIzLjY4LC0xNC41IDEyMy42OCwtMjEuNSIvPgo8L2c+CjwhLS0gY2xpbWIgLS0+
CjxnIGlkPSJub2RlMyIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+Y2xpbWI8L3RpdGxlPgo8ZWxsaXBz
ZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iNDQ3LjIzIiBjeT0iLTE4IiByeD0iODAu
NzkiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI0NDcuMjMiIHk9Ii0x
NS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFj
ZSIgZm9udC1zaXplPSIxMC4wMCI+UiA9IENsaW1iKG5leHRfcHJlYyk8L3RleHQ+CjwvZz4KPCEt
LSB3aGlsZSYjNDU7Jmd0O2NsaW1iIC0tPgo8ZyBpZD0iZWRnZTIiIGNsYXNzPSJlZGdlIj4KPHRp
dGxlPndoaWxlJiM0NTsmZ3Q7Y2xpbWI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9
ImJsYWNrIiBkPSJNMzQ0LjczLC0xOEMzNDguNiwtMTggMzUyLjQ2LC0xOCAzNTYuMjksLTE4Ii8+
Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjM1Ni4zNywtMjEu
NSAzNjYuMzcsLTE4IDM1Ni4zNywtMTQuNSAzNTYuMzcsLTIxLjUiLz4KPC9nPgo8IS0tIGxub2Rl
IC0tPgo8ZyBpZD0ibm9kZTQiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmxub2RlPC90aXRsZT4KPGVs
bGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjYyMy40NiIgY3k9Ii0xOCIgcng9
IjczLjY3IiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNjIzLjQ2IiB5
PSItMTUuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25v
c3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkwgPSBOb2RlKEwsIG9wLCBSKTwvdGV4dD4KPC9nPgo8
IS0tIGNsaW1iJiM0NTsmZ3Q7bG5vZGUgLS0+CjxnIGlkPSJlZGdlMyIgY2xhc3M9ImVkZ2UiPgo8
dGl0bGU+Y2xpbWImIzQ1OyZndDtsbm9kZTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIGQ9Ik01MjcuOTYsLTE4QzUzMS44MSwtMTggNTM1LjY5LC0xOCA1MzkuNTUsLTE4
Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjUzOS43Mywt
MjEuNSA1NDkuNzMsLTE4IDUzOS43MywtMTQuNSA1MzkuNzMsLTIxLjUiLz4KPC9nPgo8L2c+Cjwv
c3ZnPgo=
" alt="" />
<figcaption>Figure 19–10: Precedence Climbing Loop</figcaption>
</figure>

**Specification:**

Implement a standalone `PrecedenceClimbingParser` class. Parse Faberge
expression tokens using the precedence climbing algorithm, and verify
that the emitted AST matches the Pratt parser's output node-for-node.

``` cpp
class PrecedenceClimbingParser {
public:
    explicit PrecedenceClimbingParser(std::vector<Token> tokens);
    std::unique_ptr<ExprAST> parse_expression(int min_prec = 1) {
        auto left = parse_primary();
        while (is_binary_op(current_token()) && get_precedence(current_token()) >= min_prec) {
            Token op = advance();
            int p = get_precedence(op);
            int next_p = is_right_assoc(op) ? p : p + 1;
            auto right = parse_expression(next_p);
            left = std::make_unique<BinaryExprAST>(op.location, op.type, std::move(left), std::move(right));
        }
        return left;
    }
private:
    std::unique_ptr<ExprAST> parse_primary();
};
```

**Verification Tasks:**

1.  **Tree Isomorphism:** Parse `1 + 2 * 3 - 4 / 5` using both the Pratt
    parser and the Precedence Climbing parser. Compare both trees and
    assert structural identity.
2.  **Associativity Verification:** Parse a sequence of
    right-associative operators `a ^ b ^ c`. Verify that Precedence
    Climbing constructs an identical right-skewed AST.
3.  **Performance Profiling:** Parse $50,000$ synthetic expressions
    across both implementations. Compare execution times and verify that
    both complete within comparable performance bands.

#### Exercise 19.12: Automated Grammar Mutation and Parser Fuzzing Engine

**Context and Theory:**

Compilers must maintain **Parsing Robustness**: no token
sequence—regardless of grammatical invalidity, structural corruption, or
nesting depth—should cause the parser to crash, trigger undefined
behaviour, or exhaust the OS stack via unbounded recursion.

An automated **Parser Fuzzing Engine** subjects the grammar parser to
adversarial testing:

- **Token Deletion:** Randomly dropping essential structural tokens
  (e.g. omitting `then` from an `if` expression, or dropping closing
  `}`).
- **Token Insertion:** Injecting arbitrary keywords or punctuation into
  expression sequences (e.g. `1 + let * 4`).
- **Pathological Nesting:** Emitting millions of opening delimiters
  `(((( ...` to test recursion depth limits.

<figure id="fig:parser-fuzzer">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI2MzdwdCIgaGVpZ2h0PSI2NnB0Igogdmlld0JveD0iMC4wMCAwLjAw
IDYzNi44NiA2NS45OCIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIiB4bWxuczp4
bGluaz0iaHR0cDovL3d3dy53My5vcmcvMTk5OS94bGluayI+CjxnIGlkPSJncmFwaDAiIGNsYXNz
PSJncmFwaCIgdHJhbnNmb3JtPSJzY2FsZSgxIDEpIHJvdGF0ZSgwKSB0cmFuc2xhdGUoNCA2MS45
OCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gdmFsaWQgLS0+CjxnIGlkPSJub2RlMSIgY2xhc3M9
Im5vZGUiPgo8dGl0bGU+dmFsaWQ8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9
ImJsYWNrIiBjeD0iNTQuNDUiIGN5PSItMjguOTkiIHJ4PSI1NC4zOSIgcnk9IjIxLjQzIi8+Cjx0
ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjU0LjQ1IiB5PSItMzEuOTkiIGZvbnQtZmFtaWx5
PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEw
LjAwIj5WYWxpZDwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNTQuNDUiIHk9
Ii0yMC45OSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25v
c3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkFTVCBUb2tlbnM8L3RleHQ+CjwvZz4KPCEtLSBtdXRh
dG9yIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRpdGxlPm11dGF0b3I8L3RpdGxl
Pgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMjI3Ljc3IiBjeT0iLTI4
Ljk5IiByeD0iOTYuNzUiIHJ5PSIyMS40MyIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4
PSIyMjcuNzciIHk9Ii0zMS45OSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2Fu
cyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlsgR3JhbW1hciBNdXRhdG9yIF08
L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjIyNy43NyIgeT0iLTIwLjk5IiBm
b250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9u
dC1zaXplPSIxMC4wMCI+KERyb3AsIEluc2VydCwgU3dhcCk8L3RleHQ+CjwvZz4KPCEtLSB2YWxp
ZCYjNDU7Jmd0O211dGF0b3IgLS0+CjxnIGlkPSJlZGdlMSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+
dmFsaWQmIzQ1OyZndDttdXRhdG9yPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJi
bGFjayIgZD0iTTEwOS4xOCwtMjguOTlDMTEyLjg5LC0yOC45OSAxMTYuNjgsLTI4Ljk5IDEyMC41
MywtMjguOTkiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0i
MTIwLjg1LC0zMi40OSAxMzAuODUsLTI4Ljk5IDEyMC44NSwtMjUuNDkgMTIwLjg1LC0zMi40OSIv
Pgo8L2c+CjwhLS0gbXV0YXRlZCAtLT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRs
ZT5tdXRhdGVkPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9
IjM4OC4zNiIgY3k9Ii0yOC45OSIgcng9IjQxLjk0IiByeT0iMjguOTgiLz4KPHRleHQgdGV4dC1h
bmNob3I9Im1pZGRsZSIgeD0iMzg4LjM2IiB5PSItMzcuNDkiIGZvbnQtZmFtaWx5PSJDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5NdXRh
dGVkPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIzODguMzYiIHk9Ii0yNi40
OSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2Ui
IGZvbnQtc2l6ZT0iMTAuMDAiPlRva2VuPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxl
IiB4PSIzODguMzYiIHk9Ii0xNS40OSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUg
U2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlN0cmVhbTwvdGV4dD4KPC9n
Pgo8IS0tIG11dGF0b3ImIzQ1OyZndDttdXRhdGVkIC0tPgo8ZyBpZD0iZWRnZTIiIGNsYXNzPSJl
ZGdlIj4KPHRpdGxlPm11dGF0b3ImIzQ1OyZndDttdXRhdGVkPC90aXRsZT4KPHBhdGggZmlsbD0i
bm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTMyNC42OCwtMjguOTlDMzI4LjU4LC0yOC45OSAzMzIu
NDMsLTI4Ljk5IDMzNi4xOCwtMjguOTkiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0i
YmxhY2siIHBvaW50cz0iMzM2LjQ1LC0zMi40OSAzNDYuNDUsLTI4Ljk5IDMzNi40NSwtMjUuNDkg
MzM2LjQ1LC0zMi40OSIvPgo8L2c+CjwhLS0gcGFyc2VyIC0tPgo8ZyBpZD0ibm9kZTQiIGNsYXNz
PSJub2RlIj4KPHRpdGxlPnBhcnNlcjwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIGN4PSI1NDAuNDciIGN5PSItMjguOTkiIHJ4PSI4OC4yOCIgcnk9IjIxLjQzIi8+
Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjU0MC40NyIgeT0iLTMxLjk5IiBmb250LWZh
bWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXpl
PSIxMC4wMCI+WyBGYWJlcmdlIFBhcnNlciBdPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIiB4PSI1NDAuNDciIHk9Ii0yMC45OSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWph
VnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPihNdXN0IE5PVCBDcmFz
aCEpPC90ZXh0Pgo8L2c+CjwhLS0gbXV0YXRlZCYjNDU7Jmd0O3BhcnNlciAtLT4KPGcgaWQ9ImVk
Z2UzIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5tdXRhdGVkJiM0NTsmZ3Q7cGFyc2VyPC90aXRsZT4K
PHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTQzMC4zOCwtMjguOTlDNDMzLjk4
LC0yOC45OSA0MzcuNywtMjguOTkgNDQxLjUyLC0yOC45OSIvPgo8cG9seWdvbiBmaWxsPSJibGFj
ayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSI0NDEuOCwtMzIuNDkgNDUxLjgsLTI4Ljk5IDQ0MS44
LC0yNS40OSA0NDEuOCwtMzIuNDkiLz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 19–11: Parser Fuzzing Engine</figcaption>
</figure>

**Specification:**

Construct a `ParserFuzzEngine` class. The harness takes valid Faberge
programs, applies stochastic grammatical mutations (token drops,
delimiter swaps, operator insertions), passes the corrupted tokens to
the `Parser`, and asserts that the parser synchronises or reports errors
cleanly without halting abnormally.

``` cpp
class ParserFuzzEngine {
public:
    struct FuzzStats {
        size_t programs_tested{0};
        size_t errors_handled{0};
        bool survived{true};
    };
    static FuzzStats run_grammar_fuzz(std::string_view seed_source, size_t iterations) {
        FuzzStats stats;
        std::mt19937_64 rng(42);
        Lexer base_lexer(seed_source);
        auto base_tokens = base_lexer.tokenise();

        for (size_t i = 0; i < iterations; ++i) {
            auto mutated = mutate_tokens(base_tokens, rng);
            Parser parser(std::move(mutated), seed_source);
            auto ast = parser.parse_program(); // Must NEVER segfault or throw unhandled exceptions
            stats.errors_handled += parser.error_count();
            ++stats.programs_tested;
        }
        return stats;
    }
private:
    static std::vector<Token> mutate_tokens(std::vector<Token> tokens, std::mt19937_64& rng);
};
```

**Verification Tasks:**

1.  **Random Mutation Resistance:** Subject the parser to $10,000$
    randomized token mutations of a complex recursive Faberge script.
    Verify that the parser halts with clean error reporting in $100\%$
    of test runs.
2.  **Pathological Recursion Clamping:** Generate an adversarial token
    stream consisting of $10,000$ nested opening parentheses `((((...`.
    Verify that the parser catches the maximum recursion limit and halts
    gracefully rather than overflowing the stack.
3.  **Delimiter Mismatch Recovery:** Randomly replace closing
    parentheses with brackets or braces (e.g. `(1 + 2]`). Verify that
    the parser identifies delimiter mismatches and continues processing
    subsequent lines.